# 10 — ROI generator alternatives and Medium topology ablations (LEVIR-CD)

**Questions (exploratory, validation only):**

1. Do ROI geometry / post-processing choices explain the Medium Full crop-workload advantage?
2. Can other crop proposal strategies outperform the project's current `satchange.semantic.roi.extract_rois` at **similar change coverage**?
3. Can post-processing the *binary mask* reduce Medium's merging problems and improve strict one-to-one Component Recall without disproportionate Pixel F1 loss?
4. Is high ROI coverage genuinely location-specific, compared with **equal-budget random-position crops**?

**Two distinct intervention types:** ROI generators change *where crops are sent* while leaving the binary prediction unchanged. Morphology/watershed changes *the binary prediction itself*, and may then change ROIs. They are evaluated separately, with an optional interaction experiment at the end.

**Scientific guardrails:** H1–H9 have already used LEVIR-CD test. This notebook uses **only validation** and splits its 64 pairs into development and internal audit halves. Comparisons remain exploratory; audit is a guard against selecting and reporting on the exact same images, not an untouched external test set. No training, GPU inference, model checkpoints, or SECOND labels are required; all scores come from the previously verified H1–H5 cache. The original 06 metrics, score accessor, ground truth, and project ROI code are reused without reimplementation.

## 0. One configuration cell — Run All executes every selected experiment

Default settings run ROI alternatives, topology diagnostics, an optional watershed split (if scikit-image is installed), a compact interaction study, bootstrap comparisons, visualizations and a ZIP export. To speed up execution, set `RUN_MODE = "quick"`; this **reduces experiment coverage** and is clearly reported in the manifest. Never enable test evaluation in this notebook.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('.')
V1_RUN = 'H1_H5_v1'
OUT_NAME = 'H10_roi_generator_topology_ablations'
RUN_MODE = 'standard'  # 'standard' or 'quick'
RUN_ROI_ALTERNATIVES = True
RUN_TOPOLOGY_ABLATIONS = True
RUN_INTERACTION = True
RUN_CASE_STUDIES = True
RUN_WATERSHED = True  # Skips clearly if skimage unavailable; no pip install is attempted.

# Fixed operating points; this is NOT a new HPO search.
ROI_THRESHOLDS = (0.35, 0.70)
TOPOLOGY_THRESHOLDS = (0.35, 0.70, 0.95)
ROI_MODELS = ('medium_full','light_full','medium_resized','light_resized')
TOPOLOGY_MODELS = ('medium_full','light_full','medium_resized','light_resized')
TARGET_COVERAGE_95 = {'full': 0.975, 'resized': 0.935}
COVERAGE_MATCH_TOLERANCE = 0.005
INTERNAL_AUDIT_SEED = 1029
BOOTSTRAP_REPS = 2000
BOOTSTRAP_SEED = 20301029
CONTEXT_PX = 0
MIN_COMPONENT_AREA = 1
# Proposals built from the source project's minimum crop size, and alternative minimum sizes.
BASE_CROP_SIZE = 128
SMALL_CROP_SIZE = 64
# Merge only when the overlap fraction of the smaller crop reaches this value.
MERGE_OVERLAP_FRACTION = 0.30
# Prevent huge merged crops: fraction of a native image, not an arbitrary fixed pixel cap.
MAX_MERGED_AREA_FRACTION = 0.25
# For the random-position control, positions are seeded independently per image/policy.
RANDOM_CONTROL_REPEATS = 3
CASE_PAIR_INDICES = (5, 32, 47)

if RUN_MODE not in ('standard','quick'):
    raise ValueError('RUN_MODE must be standard or quick')
if RUN_MODE == 'quick':
    ROI_MODELS = ('medium_full','light_full')
    TOPOLOGY_MODELS = ('medium_full','light_full')
    ROI_THRESHOLDS = (0.35,)
    TOPOLOGY_THRESHOLDS = (0.35, 0.70)
    RANDOM_CONTROL_REPEATS = 1
    RUN_INTERACTION = False
print('RUN_MODE:',RUN_MODE,'| ROI:',RUN_ROI_ALTERNATIVES,
      '| topology:',RUN_TOPOLOGY_ABLATIONS,'| interaction:',RUN_INTERACTION)

RUN_MODE: standard | ROI: True | topology: True | interaction: True


## 1. Original 06 experimental core (byte-for-byte matching original protocol)

This cell is copied unchanged from H09, which itself embeds the original 06 experiment implementation. It is verified against the H1–H5 manifest signature at runtime. All score loading, ground truth and component matching come from that original core.

In [2]:
CORE_SOURCE = r'''"""Read-only LEVIR-CD inference and exploratory H1--H5 analysis.

Project adapter reviewed against the supplied levir_hypotheses_context.zip.
No training, downloads, writes to src/, or changes to final_models/.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import itertools
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
from collections import OrderedDict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from functools import lru_cache
from pathlib import Path
from typing import Any, Mapping
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import scipy
import PIL
import pandas as pd
import torch
from PIL import Image
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from torch.nn import functional as F

IMPLEMENTATION_VERSION = "levir-H1-H5-1.0.0"
VARIANTS = ("light_full", "light_resized", "medium_full", "medium_resized",
            "heavy_full", "heavy_resized")


@dataclass
class Settings:
    project_root: str
    run_name: str = "H1_H5_v1"
    dataset_root: str | None = None
    variants: tuple[str, ...] = VARIANTS
    device: str = "cuda:0"
    include_other_seeds: bool = False
    trust_own_seed_checkpoints: bool = False
    cpu_threads: int = 4
    validation_atol: float = 0.001
    connectivity: int = 2  # 2 = 8-neighbour connectivity in a 2D mask.
    size_quantiles: tuple[float, float] = (1 / 3, 2 / 3)
    iou_thresholds: tuple[float, ...] = (0.1, 0.25, 0.5)
    primary_iou: float = 0.5
    coverage_thresholds: tuple[float, ...] = (0.1, 0.25, 0.5)
    boundary_tolerances_px: tuple[int, ...] = (1, 2, 4)
    primary_boundary_px: int = 2
    recall_targets: tuple[float, ...] = (0.8, 0.9)
    recall_match_tolerance: float = 0.005
    threshold_grid: tuple[float, ...] = tuple(np.linspace(0, 1, 101).tolist())
    bootstrap_repetitions: int = 1000
    bootstrap_seed: int = 20261020
    cost_source: str = "remeasure"  # or "saved_benchmark"; CSV is never used as truth.
    benchmark_warmup: int = 20
    benchmark_repetitions: int = 100
    minimum_free_gib: float = 1.0
    # Monetary values are OPTIONAL user scenarios, not measured market prices.
    gpu_price_per_hour: float | None = None
    review_price_per_alert: float | None = None
    missed_component_price: float | None = None
    scenario_pairs: int = 1000
    minimum_object_recall: float | None = None
    maximum_false_alerts_per_pair: float | None = None
    maximum_forward_ms: float | None = None
    # A CSV mapping split,sample_id,cluster_id can replace image-pair bootstrap units.
    cluster_csv: str | None = None


def _json_value(x: Any) -> Any:
    if isinstance(x, dict):
        return {str(k): _json_value(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_json_value(v) for v in x]
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, np.ndarray):
        return _json_value(x.tolist())
    if isinstance(x, np.generic):
        return _json_value(x.item())
    if isinstance(x, float) and not math.isfinite(x):
        return None
    return x


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".{os.getpid()}.tmp")
    tmp.write_text(json.dumps(_json_value(obj), ensure_ascii=False, indent=2,
                              sort_keys=True, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def digest(obj: Any) -> str:
    return hashlib.sha256(json.dumps(_json_value(obj), sort_keys=True,
                                     ensure_ascii=False).encode()).hexdigest()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()


def safe_div(a: float, b: float) -> float:
    return float(a / b) if b > 0 else float("nan")


def metric_counts(pred: np.ndarray, gt: np.ndarray) -> dict[str, int]:
    if pred.shape != gt.shape or pred.dtype != bool or gt.dtype != bool:
        raise ValueError("Expected aligned 2-D boolean masks.")
    tp = int(np.count_nonzero(pred & gt))
    fp = int(np.count_nonzero(pred & ~gt))
    pos = int(gt.sum())
    return {"tp": tp, "fp": fp, "fn": pos - tp,
            "tn": gt.size - pos - fp}


def binary_metrics(c: Mapping[str, float]) -> dict[str, float]:
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    return {"pixel_precision": safe_div(tp, tp + fp),
            "pixel_recall": safe_div(tp, tp + fn),
            "pixel_f1": safe_div(2 * tp, 2 * tp + fp + fn),
            "pixel_iou": safe_div(tp, tp + fp + fn)}


def _tag(x: float) -> str:
    return f"{round(100 * x):03d}"


def _threshold_key(t: float) -> str:
    return hashlib.sha256(np.float32(t).tobytes()).hexdigest()[:12]


def boundary(mask: np.ndarray) -> np.ndarray:
    return mask & ~ndi.binary_erosion(mask, structure=np.ones((3, 3), bool),
                                      border_value=0)


def _distance_to(mask: np.ndarray) -> np.ndarray:
    if not mask.any():
        return np.full(mask.shape, np.inf, dtype=np.float32)
    return ndi.distance_transform_edt(~mask)


def inner_band(mask: np.ndarray, width: int) -> np.ndarray:
    return mask & ~ndi.binary_erosion(mask, structure=np.ones((3, 3), bool),
                                      iterations=width, border_value=0)


def match_components(gt_labels: np.ndarray, pred_labels: np.ndarray,
                     gt_areas: np.ndarray, pred_areas: np.ndarray,
                     iou_thresholds: tuple[float, ...]) -> tuple[dict, dict]:
    """One-to-one max-cardinality matching, then max IoU among such matchings.

    The matrix includes only IDs participating in an eligible IoU edge; tiny
    unrelated predictions still count as unmatched predictions, never disappear.
    """
    ng, npred = len(gt_areas) - 1, len(pred_areas) - 1
    overlap = (gt_labels > 0) & (pred_labels > 0)
    if overlap.any():
        codes = (gt_labels[overlap].astype(np.int64) * (npred + 1)
                 + pred_labels[overlap])
        codes, intersections = np.unique(codes, return_counts=True)
        gi, pi = codes // (npred + 1), codes % (npred + 1)
        ious = intersections / (gt_areas[gi] + pred_areas[pi] - intersections)
    else:
        gi = pi = np.array([], dtype=np.int64)
        ious = np.array([], dtype=float)
    info = {
        "gt_with_multiple_overlapping_pred_components": int(
            np.count_nonzero(np.bincount(gi, minlength=ng + 1)[1:] > 1)),
        "pred_components_overlapping_multiple_gt": int(
            np.count_nonzero(np.bincount(pi, minlength=npred + 1)[1:] > 1)),
    }
    found = {}
    for threshold in iou_thresholds:
        eligible = ious >= threshold
        gids, pids = np.unique(gi[eligible]), np.unique(pi[eligible])
        pairs = {}
        if len(gids) and len(pids):
            if len(gids) * len(pids) > 10_000_000:
                raise MemoryError("Component matching matrix too large; inspect fragmented masks.")
            weight = np.zeros((len(gids), len(pids)), dtype=np.float64)
            rr, cc = np.searchsorted(gids, gi[eligible]), np.searchsorted(pids, pi[eligible])
            # Total tie-breaking contribution < 1: one extra match always wins.
            weight[rr, cc] = 1 + ious[eligible] / (min(len(gids), len(pids)) + 1)
            r, c = linear_sum_assignment(weight, maximize=True)
            keep = weight[r, c] > 0
            for ri, ci in zip(r[keep], c[keep]):
                pairs[int(gids[ri])] = (int(pids[ci]), float(
                    (weight[ri, ci] - 1) * (min(len(gids), len(pids)) + 1)))
        found[threshold] = pairs
    return found, info


def exact_threshold_counts(scores: np.ndarray, gt: np.ndarray,
                           thresholds: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Exact counts on a predeclared grid, no lossy score histogram/rounding."""
    thresholds = thresholds.astype(np.float32)
    index = np.searchsorted(thresholds, scores.ravel(), side="right")
    positive = np.bincount(index[gt.ravel()], minlength=len(thresholds) + 1)
    negative = np.bincount(index[~gt.ravel()], minlength=len(thresholds) + 1)
    return np.cumsum(positive[::-1])[::-1][1:], np.cumsum(negative[::-1])[::-1][1:]


def pair_counts(a: np.ndarray, b: np.ndarray, gt: np.ndarray) -> dict:
    common = a & b
    union = a | b
    return {
        "n_positive": int(gt.sum()), "tp_a": int((a & gt).sum()),
        "tp_b": int((b & gt).sum()), "common_tp": int((common & gt).sum()),
        "a_only_tp": int((a & ~b & gt).sum()),
        "b_only_tp": int((b & ~a & gt).sum()),
        "common_fn": int((~union & gt).sum()),
        "fp_a": int((a & ~gt).sum()), "fp_b": int((b & ~gt).sum()),
        "common_fp": int((common & ~gt).sum()),
        "tp_or": int((union & gt).sum()), "fp_or": int((union & ~gt).sum()),
        "prediction_intersection": int(common.sum()), "prediction_union": int(union.sum()),
    }


def complementarity(c: Mapping[str, float]) -> dict:
    n, best = c["n_positive"], max(c["tp_a"], c["tp_b"])
    return {
        "jaccard_tp": safe_div(c["common_tp"], c["tp_or"]),
        "jaccard_fp": safe_div(c["common_fp"], c["fp_or"]),
        "jaccard_prediction": safe_div(c["prediction_intersection"], c["prediction_union"]),
        "recall_a": safe_div(c["tp_a"], n), "recall_b": safe_div(c["tp_b"], n),
        "precision_a": safe_div(c["tp_a"], c["tp_a"] + c["fp_a"]),
        "precision_b": safe_div(c["tp_b"], c["tp_b"] + c["fp_b"]),
        "recall_or": safe_div(c["tp_or"], n),
        "precision_or": safe_div(c["tp_or"], c["tp_or"] + c["fp_or"]),
        "extra_tp_vs_best": c["tp_or"] - best,
        "delta_recall": safe_div(c["tp_or"] - best, n),
        "complementarity_gain": safe_div(c["tp_or"] - best, n - best),
        "cg_denominator": n - best,
        "recovery_of_a_fn_by_b": safe_div(c["b_only_tp"], n - c["tp_a"]),
        "recovery_of_b_fn_by_a": safe_div(c["a_only_tp"], n - c["tp_b"]),
    }


def pareto_flags(df: pd.DataFrame, maximize: list[str], minimize: list[str]) -> np.ndarray:
    values = np.column_stack([*[df[c].to_numpy(float) for c in maximize],
                              *[-df[c].to_numpy(float) for c in minimize]])
    finite = np.isfinite(values).all(axis=1)
    flags = np.zeros(len(df), dtype=bool)
    for i in np.flatnonzero(finite):
        other = values[finite]
        flags[i] = not np.any(np.all(other >= values[i], axis=1)
                              & np.any(other > values[i], axis=1))
    return flags


class Experiment:
    def __init__(self, settings: Settings):
        self.cfg = settings
        self.root = Path(settings.project_root).expanduser().resolve()
        if not (self.root / "src/satchange/models/factory.py").is_file():
            raise FileNotFoundError(f"Not the project root: {self.root}")
        if not re.fullmatch(r"[A-Za-z0-9_-]+", settings.run_name):
            raise ValueError("run_name must contain only ASCII letters, digits, '-' and '_'.")
        if not settings.variants or len(set(settings.variants)) != len(settings.variants):
            raise ValueError("Select a non-empty set of distinct variants.")
        for name in ("iou_thresholds", "coverage_thresholds", "recall_targets"):
            values = getattr(settings, name)
            if not values or any(not math.isfinite(x) or not 0 < x <= 1 for x in values):
                raise ValueError(f"{name} must contain values in (0,1].")
            if len({_tag(x) for x in values}) != len(values):
                raise ValueError(f"{name}: values need distinct integer-percentage labels.")
        if settings.cpu_threads < 1 or settings.scenario_pairs < 1:
            raise ValueError("cpu_threads and scenario_pairs must be positive.")
        if not 0 < settings.size_quantiles[0] <= settings.size_quantiles[1] < 1:
            raise ValueError("Invalid size quantiles.")
        if any(r < 1 or int(r) != r for r in settings.boundary_tolerances_px):
            raise ValueError("Boundary tolerances must be positive integer native pixels.")
        if settings.connectivity not in (1, 2):
            raise ValueError("connectivity must be 1 (4 neighbours) or 2 (8 neighbours).")
        if settings.primary_iou not in settings.iou_thresholds:
            raise ValueError("primary_iou must be in iou_thresholds.")
        if settings.primary_boundary_px not in settings.boundary_tolerances_px:
            raise ValueError("primary_boundary_px must be in boundary_tolerances_px.")
        if settings.cost_source not in ("remeasure", "saved_benchmark"):
            raise ValueError("cost_source must be remeasure or saved_benchmark.")
        if settings.include_other_seeds and not settings.trust_own_seed_checkpoints:
            raise ValueError("Full seed checkpoints require explicit trust_own_seed_checkpoints=True.")
        if settings.bootstrap_repetitions < 100:
            warnings.warn("Fewer than 100 bootstrap replicates: debugging only, not a stable CI.")
        if settings.benchmark_repetitions < 1 or settings.benchmark_warmup < 0:
            raise ValueError("Invalid benchmark repetition counts.")
        for name in ("gpu_price_per_hour", "review_price_per_alert", "missed_component_price"):
            v = getattr(settings, name)
            if v is not None and (not math.isfinite(v) or v < 0):
                raise ValueError(f"{name} must be nonnegative or None.")
        self.device = torch.device(settings.device)
        if self.device.type == "cuda":
            if not torch.cuda.is_available():
                raise RuntimeError("CUDA unavailable. Choose the project's GPU kernel or set device='cpu'.")
            if self.device.index is None:
                self.device = torch.device("cuda:0")
            if self.device.index >= torch.cuda.device_count():
                raise ValueError("Requested CUDA device is not visible.")
        torch.set_num_threads(settings.cpu_threads)
        src = str(self.root / "src")
        if src not in sys.path:
            sys.path.insert(0, src)
        import satchange
        loaded = Path(satchange.__file__).resolve()
        if not loaded.is_relative_to(self.root / "src"):
            raise RuntimeError(f"Wrong satchange imported from {loaded}. Restart kernel in Research I.")
        from satchange.data.levir_cd import LEVIRCDDataset
        from satchange.models import create_change_model
        from satchange.final_benchmark import _runtime, _load_checkpoint, _instantiate
        self.Dataset = LEVIRCDDataset
        self.create_model = create_change_model
        self.runtime = _runtime  # Reuse the project's FP32/TF32 runtime controls.
        self.project_load_checkpoint = _load_checkpoint
        self.project_instantiate = _instantiate
        self.dataset_root = Path(settings.dataset_root or self.root / "data/LEVIR-CD").expanduser().resolve()
        self.datasets = {s: self.Dataset(self.dataset_root, s, None, transform=None)
                         for s in ("val", "test")}
        self.out = self.root / "artifacts/hypothesis_tests" / settings.run_name
        self.reports = self.out / "reports"
        self.reports.mkdir(parents=True, exist_ok=True)
        self.cache = self.out / "cache"
        self.cache.mkdir(exist_ok=True)
        self.gt_structure = ndi.generate_binary_structure(2, settings.connectivity)
        self.specs = self._specs()
        self.spec_by_id = {s["run_id"]: s for s in self.specs}
        self.primary = [s for s in self.specs if s["primary"]]
        self.source_manifest = {
            str(p.relative_to(self.root)): sha256_file(p)
            for p in sorted((self.root / "src/satchange").rglob("*.py"))}
        self.cluster_map = self._clusters()
        print("Hashing validation/test inputs for cache integrity (no test metrics)...", flush=True)
        self.data_manifest = self._data_manifest()
        self.protocol = {
            "implementation": IMPLEMENTATION_VERSION,
            "implementation_sha256": globals().get("IMPLEMENTATION_SHA256") or (sha256_file(Path(__file__)) if "__file__" in globals() else IMPLEMENTATION_VERSION),
            "settings": asdict(settings), "sources": self.source_manifest,
            "specs": self.specs, "data_manifest": self.data_manifest,
            "cluster_map": self.cluster_map,
            "environment": {"python": platform.python_version(), "torch": str(torch.__version__),
                            "numpy": np.__version__, "pandas": pd.__version__,
                            "scipy": scipy.__version__, "Pillow": PIL.__version__,
                            "cudnn": torch.backends.cudnn.version(),
                            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
                            "cuda": torch.version.cuda, "device": str(self.device),
                            "gpu": torch.cuda.get_device_name(self.device)
                            if self.device.type == "cuda" else None},
            "common_grid": "native GT; float32 sigmoid scores; bilinear upsample before >= threshold; align_corners=False",
            "scope": "post-hoc exploratory inference; no new architecture-causal claim",
        }
        self.signature = digest(self.protocol)
        manifest_path = self.out / "run_manifest.json"
        if manifest_path.exists() and read_json(manifest_path)["signature"] != self.signature:
            raise RuntimeError("Run inputs/code/settings changed. Choose a NEW run_name; old caches are preserved.")
        if not manifest_path.exists():
            write_json(manifest_path, {"signature": self.signature, "protocol": self.protocol,
                                      "created_utc": datetime.now(timezone.utc).isoformat()})
        self._audit()
        self.plan = read_json(self.out / "protocol_frozen.json") if (self.out / "protocol_frozen.json").exists() else None
        print(f"Output: {self.out}\nModels/checkpoints: {len(self.specs)}; primary pipelines: {len(self.primary)}")
        self._storage_check()

    def _specs(self) -> list[dict]:
        result = []
        for variant in self.cfg.variants:
            if variant not in VARIANTS:
                raise ValueError(f"Unknown variant: {variant}")
            folder = self.root / "artifacts/final_models" / variant
            meta = read_json(folder / "metadata.json")
            val = read_json(folder / "validation_results.json")
            if not {"model_name", "model_parameters", "image_size", "deployment_seed", "deployment_threshold"} <= meta.keys():
                raise ValueError(f"Incomplete metadata: {folder}")
            dep = int(meta["deployment_seed"])
            rows = {int(r["seed"]): r for r in val["results"]}
            available = sorted(int(p.name.split("_")[1]) for p in folder.glob("seed_*_best.pt"))
            seeds = [dep] + ([s for s in available if s != dep] if self.cfg.include_other_seeds else [])
            for seed in seeds:
                primary = seed == dep
                file = folder / ("deployment_weights.pt" if primary else f"seed_{seed}_best.pt")
                if not file.is_file():
                    raise FileNotFoundError(file)
                if seed not in rows:
                    raise ValueError(f"No validation record for {variant} seed {seed}")
                row = rows[seed]
                threshold = float(meta["deployment_threshold"] if primary else row["validation_threshold"])
                if not 0 < threshold < 1:
                    raise ValueError("Saved thresholds must be in (0,1).")
                if abs(threshold - float(row["validation_threshold"])) > 1e-9:
                    raise ValueError(f"Threshold conflict in {variant}/{seed}")
                image_size = meta["image_size"]  # Respect Medium Full's stored 1024, not an invented None.
                if variant.endswith("resized") and image_size != 256:
                    raise ValueError(f"Unexpected resized geometry for {variant}")
                if variant.endswith("full") and image_size not in (None, 1024):
                    raise ValueError(f"Unexpected full geometry for {variant}")
                result.append({"variant": variant, "run_id": f"{variant}__s{seed}",
                    "family": variant.split("_")[0], "resolution": variant.split("_")[1],
                    "seed": seed, "primary": primary, "image_size": image_size,
                    "model_name": meta["model_name"], "model_parameters": meta["model_parameters"],
                    "saved_threshold": threshold, "weight_file": str(file),
                    "weight_sha256": sha256_file(file),
                    "metadata_sha256": sha256_file(folder / "metadata.json"),
                    "validation_sha256": sha256_file(folder / "validation_results.json"),
                    "benchmark_sha256": sha256_file(folder / "benchmark.json"),
                    "expected_validation_f1": row["validation_f1"],
                    "expected_validation_iou": row["validation_iou"],
                    "expected_validation_pairs": val.get("number_of_pairs"),
                    "available_seed_files": available})
        return result

    def _clusters(self) -> dict:
        if self.cfg.cluster_csv is None:
            return {}
        path = Path(self.cfg.cluster_csv).expanduser()
        df = pd.read_csv(path, dtype=str)
        if not {"split", "sample_id", "cluster_id"} <= set(df):
            raise ValueError("cluster_csv needs split,sample_id,cluster_id columns.")
        if df.duplicated(["split", "sample_id"]).any() or df.isna().any().any():
            raise ValueError("Duplicate or missing cluster identifiers.")
        mapping = {f"{r.split}/{r.sample_id}": r.cluster_id for r in df.itertuples()}
        for split, ds in self.datasets.items():
            for sid in ds.sample_ids:
                if f"{split}/{sid}" not in mapping:
                    raise ValueError(f"Missing cluster for {split}/{sid}")
        return mapping

    def _data_manifest(self) -> dict:
        result = {}
        pair_hashes = {}
        for split, ds in self.datasets.items():
            result[split] = []
            pair_hashes[split] = set()
            for sid in ds.sample_ids:
                row = {"sample_id": sid}
                hashes = []
                for key, files in (("A", ds.before_files), ("B", ds.after_files), ("label", ds.label_files)):
                    p = files[sid]
                    h = sha256_file(p)
                    row[key] = {"file": str(p), "bytes": p.stat().st_size, "sha256": h}
                    hashes.append(h)
                pair_hashes[split].add(tuple(hashes[:2]))
                result[split].append(row)
        if pair_hashes["val"] & pair_hashes["test"]:
            raise ValueError("Identical RGB pairs detected in validation and test. Resolve split leakage first.")
        return result

    def _audit(self) -> None:
        table = pd.DataFrame([{k: s[k] for k in ("run_id", "variant", "seed", "primary", "image_size",
            "model_name", "saved_threshold", "available_seed_files")} | {"parameters": json.dumps(s["model_parameters"])}
                                for s in self.specs])
        table.to_csv(self.reports / "inventory.csv", index=False)
        issues = []
        for family in ("light", "medium", "heavy"):
            group = [s for s in self.primary if s["family"] == family]
            if len(group) == 2 and group[0]["model_parameters"] != group[1]["model_parameters"]:
                issues.append(f"{family}: Full and Resized have different architecture parameters. H1 is a pipeline comparison, not a pure resolution ablation.")
        for s in self.primary:
            if len(s["available_seed_files"]) < 2:
                issues.append(f"{s['variant']}: only one seed file; no estimate of training-seed variance.")
        csv = self.root / "artifacts/final_models/final_models_metrics.csv"
        if csv.exists():
            original = pd.read_csv(csv)
            original.to_csv(self.reports / "historical_metrics_csv_NOT_USED.csv", index=False)
            if "f1_key" in original and not original["f1_key"].eq("test_f1").all():
                issues.append("final_models_metrics.csv mixes validation/test fields. It is NOT used for evaluation or model ranking.")
        old_cost = self.root / "artifacts/final_models/final_models_gpu_cost.csv"
        costs = []
        if old_cost.exists():
            c = pd.read_csv(old_cost)
            for s in self.primary:
                b = read_json(self.root / "artifacts/final_models" / s["variant"] / "benchmark.json")
                row = c.loc[c["model"] == s["variant"]]
                costs.append({"variant": s["variant"], "benchmark_json_ms": b.get("latency_ms_per_pair"),
                    "cost_csv_ms": float(row.iloc[0]["latency_ms"]) if len(row) else np.nan})
            pd.DataFrame(costs).to_csv(self.reports / "historical_cost_discrepancies.csv", index=False)
            issues.append("Saved cost CSV and per-model benchmark JSON are separate measurements; do not mix their protocols. H5 defaults to one fresh common forward-only benchmark.")
        issues += ["Connected GT components are change regions, NOT guaranteed building-instance labels.",
                   "Earlier test scores were already inspected: these analyses are exploratory/post-hoc.",
                   "A deployment checkpoint is not an independent replicate of its source seed.",
                   "No automatic hypothesis-proved labels are produced."]
        write_json(self.reports / "audit_notes.json", issues)
        print("\n".join("NOTE: " + x for x in issues), flush=True)

    def _storage_check(self) -> None:
        # Conservative native-cache estimate assuming 1024x1024 native images.
        n = sum(len(ds) for ds in self.datasets.values())
        bytes_needed = sum(n * 4 * ((s["image_size"] or 1024) ** 2) for s in self.specs)
        free = shutil.disk_usage(self.out).free
        print(f"Score cache upper estimate: {bytes_needed / 2**30:.2f} GiB; free disk: {free / 2**30:.1f} GiB.")
        if free < bytes_needed + self.cfg.minimum_free_gib * 2**30:
            warnings.warn("Free disk may be insufficient for all caches. Existing cache files are not deducted from this conservative estimate.")

    def _load_model(self, s: dict) -> torch.nn.Module:
        path = Path(s["weight_file"])
        if sha256_file(path) != s["weight_sha256"]:
            raise RuntimeError(f"Weights changed during run: {path}")
        if s["primary"]:
            state = torch.load(path, map_location="cpu", weights_only=True)
            if not isinstance(state, Mapping) or not state or not all(
                isinstance(v, torch.Tensor) for v in state.values()):
                raise TypeError("deployment_weights.pt must be a plain tensor state_dict.")
        else:
            # Only explicitly trusted, locally produced full checkpoints.
            state, cp = self.project_load_checkpoint(path)
            config = cp.get("parameters", cp.get("config", {})) or {}
            actual_seed = cp.get("seed", config.get("seed"))
            if actual_seed is not None and int(actual_seed) != s["seed"]:
                raise ValueError("Checkpoint seed mismatch.")
            actual_model = cp.get("model_parameters", config.get("model_parameters"))
            if actual_model is not None:
                # Compare recorded values, not invented constructor defaults.
                for key, val in actual_model.items():
                    if key in s["model_parameters"] and _json_value(val) != _json_value(s["model_parameters"][key]):
                        raise ValueError(f"Architecture conflict: {s['run_id']} {key}")
        # Reviewed project helper: no external pretraining download; strict=True.
        return self.project_instantiate(s["model_name"], s["model_parameters"], state)

    def _release(self) -> None:
        gc.collect()
        if self.device.type == "cuda":
            with torch.cuda.device(self.device):
                torch.cuda.empty_cache()

    def _sync(self) -> None:
        if self.device.type == "cuda":
            torch.cuda.synchronize(self.device)

    def _native_path(self, s: dict, split: str, index: int) -> Path:
        return self.cache / s["run_id"] / split / f"{index:05d}.npy"

    def _native_score(self, s: dict, split: str, index: int) -> np.ndarray:
        p = self._native_path(s, split, index)
        if not p.is_file():
            raise FileNotFoundError(f"Run inference for {s['run_id']} {split}: {p}")
        a = np.load(p, allow_pickle=False)
        if a.dtype != np.float32 or a.ndim != 2 or not np.isfinite(a).all():
            raise ValueError(f"Invalid cached score file: {p}")
        return a

    def score(self, s: dict, split: str, index: int) -> np.ndarray:
        a = self._native_score(s, split, index)
        shape = self.gt(split, index)["mask"].shape
        if a.shape != shape:
            with torch.inference_mode():
                a = F.interpolate(torch.from_numpy(a)[None, None], size=shape,
                                  mode="bilinear", align_corners=False)[0, 0].numpy()
        return a

    @lru_cache(maxsize=2)
    def gt(self, split: str, index: int) -> dict:
        ds = self.datasets[split]
        sid = ds.sample_ids[index]
        with Image.open(ds.label_files[sid]) as im:
            mask = np.array(im.convert("L"), dtype=np.uint8) > 0
        labels, ng = ndi.label(mask, structure=self.gt_structure)
        areas = np.bincount(labels.ravel(), minlength=ng + 1)
        objects = ndi.find_objects(labels)
        border_ids = set(np.unique(np.concatenate((labels[0], labels[-1], labels[:, 0], labels[:, -1]))))
        b = boundary(mask)
        return {"sample_id": sid, "mask": mask, "labels": labels, "areas": areas,
                "n_gt": ng, "slices": objects, "border_ids": border_ids,
                "boundary": b, "distance": _distance_to(b),
                "inner_band": inner_band(mask, self.cfg.primary_boundary_px)}

    def run_variant(self, variant: str, split: str = "val") -> None:
        if split not in ("val", "test"):
            raise ValueError(split)
        if split == "test" and self.plan is None:
            raise RuntimeError("Freeze the validation protocol before test inference.")
        matching = [s for s in self.specs if s["variant"] == variant]
        if not matching:
            raise ValueError(f"Variant not enabled: {variant}")
        for s in matching:
            self._run_checkpoint(s, split)

    def _run_checkpoint(self, s: dict, split: str) -> None:
        ds = self.Dataset(self.dataset_root, split, s["image_size"], transform=None)
        if ds.sample_ids != self.datasets[split].sample_ids:
            raise ValueError("Sample IDs/order differ between model and native evaluation.")
        folder = self.cache / s["run_id"] / split
        folder.mkdir(parents=True, exist_ok=True)
        completed = folder / "complete.json"
        if completed.exists():
            info = read_json(completed)
            if info["signature"] != self.signature:
                raise RuntimeError("Stale score cache.")
            if all(self._native_path(s, split, i).is_file() for i in range(len(ds))):
                print(f"CACHE {s['run_id']} {split}: {len(ds)} pairs")
                return
        model = None
        count = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
        parameters = None
        t0 = time.perf_counter()
        try:
            with self.runtime(self.device), torch.inference_mode():
                for i in range(len(ds)):
                    p = self._native_path(s, split, i)
                    sample = ds[i]
                    if p.exists():
                        probabilities = self._native_score(s, split, i)
                    else:
                        if shutil.disk_usage(self.out).free < self.cfg.minimum_free_gib * 2**30:
                            raise OSError("Disk reserve reached. Completed scores preserved; free space and resume.")
                        if model is None:
                            model = self._load_model(s).to(self.device)
                            parameters = sum(p.numel() for p in model.parameters())
                        a = sample["before"][None].to(self.device, dtype=torch.float32)
                        b = sample["after"][None].to(self.device, dtype=torch.float32)
                        logits = model(a, b)
                        if not isinstance(logits, torch.Tensor) or tuple(logits.shape) != (1, 1, *a.shape[-2:]):
                            raise ValueError(f"Unexpected output shape for {s['run_id']}")
                        probabilities = torch.sigmoid(logits)[0, 0].float().cpu().numpy()
                        if not np.isfinite(probabilities).all():
                            raise ValueError("Non-finite predictions.")
                        tmp = p.with_suffix(".tmp")
                        with tmp.open("wb") as f:
                            np.save(f, probabilities, allow_pickle=False)
                        tmp.replace(p)
                        del a, b, logits
                    gt = sample["mask"][0].numpy() >= 0.5
                    if probabilities.shape != gt.shape:
                        raise ValueError("Native output and processed target do not align.")
                    for k, v in metric_counts(probabilities >= np.float32(s["saved_threshold"]), gt).items():
                        count[k] += v
                    if i == 0 or (i + 1) % 16 == 0 or i + 1 == len(ds):
                        print(f"{s['run_id']:25s} {split}: {i+1}/{len(ds)}", flush=True)
            result = {"signature": self.signature, "run_id": s["run_id"], "split": split,
                      "n_pairs": len(ds), "saved_threshold": s["saved_threshold"],
                      "native_counts": count, "native_metrics": binary_metrics(count),
                      "parameters": parameters, "elapsed_with_io_and_cache_seconds": time.perf_counter() - t0}
            write_json(completed, result)
        finally:
            del model
            self._release()

    def _reproduction(self) -> pd.DataFrame:
        rows = []
        for s in self.specs:
            p = self.cache / s["run_id"] / "val/complete.json"
            if not p.exists():
                raise RuntimeError(f"Validation inference incomplete: {s['run_id']}")
            r = read_json(p)
            if r["signature"] != self.signature:
                raise RuntimeError("Stale validation cache.")
            n_ok = s["expected_validation_pairs"] in (None, r["n_pairs"])
            df1 = r["native_metrics"]["pixel_f1"] - s["expected_validation_f1"]
            diou = r["native_metrics"]["pixel_iou"] - s["expected_validation_iou"]
            rows.append({"run_id": s["run_id"], "pairs": r["n_pairs"],
                "native_f1": r["native_metrics"]["pixel_f1"], "saved_f1": s["expected_validation_f1"],
                "native_iou": r["native_metrics"]["pixel_iou"], "saved_iou": s["expected_validation_iou"],
                "delta_f1": df1, "delta_iou": diou,
                "passed": n_ok and abs(df1) <= self.cfg.validation_atol and abs(diou) <= self.cfg.validation_atol})
        df = pd.DataFrame(rows)
        df.to_csv(self.reports / "validation_reproduction.csv", index=False)
        if not df["passed"].all():
            raise RuntimeError("Validation reproduction failed. Inspect validation_reproduction.csv. Do not tune on test or bypass this check blindly.")
        return df

    def _curve(self, s: dict) -> pd.DataFrame:
        grid = np.unique(np.array([*self.cfg.threshold_grid, s["saved_threshold"]], dtype=np.float32))
        if np.any(np.diff(grid) <= 0) or grid[0] < 0 or grid[-1] > 1:
            raise ValueError("Thresholds must be unique and in [0,1].")
        tp, fp = np.zeros(len(grid), np.int64), np.zeros(len(grid), np.int64)
        pos = neg = 0
        for i in range(len(self.datasets["val"])):
            g = self.gt("val", i)["mask"]
            p, n = exact_threshold_counts(self.score(s, "val", i), g, grid)
            tp += p; fp += n
            pos += int(g.sum()); neg += int((~g).sum())
        records = []
        for t, p, f in zip(grid, tp, fp):
            c = {"tp": int(p), "fp": int(f), "fn": pos-int(p), "tn": neg-int(f)}
            records.append({"run_id": s["run_id"], "threshold": float(t), **c, **binary_metrics(c)})
        return pd.DataFrame(records)

    @staticmethod
    def _at_recall(curve: pd.DataFrame, target: float) -> dict:
        feasible = curve.loc[curve["pixel_recall"] >= target - 1e-12].copy()
        if feasible.empty:
            return {"attainable": False, "target_recall": target, "threshold": None}
        # Minimal recall overshoot, then best precision, then highest threshold.
        row = feasible.sort_values(["pixel_recall", "pixel_precision", "threshold"],
                                   ascending=[True, False, False], na_position="last").iloc[0]
        return {"attainable": True, "target_recall": target, "threshold": float(row["threshold"]),
                "validation_recall": float(row["pixel_recall"]),
                "validation_precision": float(row["pixel_precision"])}

    def _make_operations(self, curves: dict[str, pd.DataFrame]) -> dict:
        operations = {}
        for s in self.specs:
            curve = curves[s["run_id"]]
            best = curve.sort_values(["pixel_f1", "pixel_precision", "threshold"],
                                     ascending=[False, False, False], na_position="last").iloc[0]
            ops = {"saved": {"threshold": s["saved_threshold"], "attainable": True},
                   "val_f1": {"threshold": float(best["threshold"]), "attainable": True,
                              "validation_f1": float(best["pixel_f1"])}}
            for r in self.cfg.recall_targets:
                ops[f"recall_{_tag(r)}"] = self._at_recall(curve, r)
            operations[s["run_id"]] = ops
        return operations

    def _size_cutoffs(self) -> list[float]:
        areas = np.concatenate([self.gt("val", i)["areas"][1:]
                                for i in range(len(self.datasets["val"]))])
        if not len(areas):
            raise ValueError("No validation GT components.")
        cuts = np.quantile(areas, self.cfg.size_quantiles).tolist()
        if cuts[0] == cuts[1]:
            warnings.warn("Tied size cutoffs: some bins can be empty; these metrics remain undefined.")
        return cuts

    def _size(self, area: int, cuts: list[float]) -> str:
        return "small" if area <= cuts[0] else "medium" if area <= cuts[1] else "large"

    def _evaluate_mask(self, pred: np.ndarray, gt: dict, cuts: list[float],
                       geometry: bool = True) -> tuple[dict, list[dict]]:
        g, gl = gt["mask"], gt["labels"]
        pl, npred = ndi.label(pred, structure=self.gt_structure)
        pa = np.bincount(pl.ravel(), minlength=npred + 1)
        matched, overlap_info = match_components(gl, pl, gt["areas"], pa, self.cfg.iou_thresholds)
        inside = np.bincount(gl[pred].ravel(), minlength=gt["n_gt"] + 1)
        band_pixels = np.bincount(gl[gt["inner_band"]].ravel(), minlength=gt["n_gt"] + 1)
        row = {"sample_id": gt["sample_id"], **metric_counts(pred, g),
               "n_gt": gt["n_gt"], "n_pred": npred, **overlap_info}
        for t, pairs in matched.items():
            tag = _tag(t)
            row.update({f"obj_tp_{tag}": len(pairs), f"obj_fp_{tag}": npred-len(pairs),
                        f"obj_fn_{tag}": gt["n_gt"]-len(pairs)})
        for name in ("small", "medium", "large"):
            row.update({f"{name}_gt": 0, f"{name}_tp_pixels": 0,
                        f"{name}_positive_pixels": 0, f"{name}_obj_tp": 0,
                        f"{name}_nonborder_gt": 0, f"{name}_nonborder_obj_tp": 0})
        row["completely_missed_gt"] = int(np.count_nonzero(inside[1:] == 0))
        for t in self.cfg.coverage_thresholds:
            coverage = inside[1:] / gt["areas"][1:]
            row[f"coverage_hits_{_tag(t)}"] = int(np.count_nonzero(coverage >= t))
        if geometry:
            pb = boundary(pred)
            pdistance = _distance_to(pb)
            row.update(boundary_pred_count=int(pb.sum()), boundary_gt_count=int(gt["boundary"].sum()))
            for r in self.cfg.boundary_tolerances_px:
                row[f"boundary_matched_pred_{r}"] = int(np.count_nonzero(pb & (gt["distance"] <= r)))
                row[f"boundary_matched_gt_{r}"] = int(np.count_nonzero(gt["boundary"] & (pdistance <= r)))
            band = inner_band(pred, self.cfg.primary_boundary_px)
            row["boundary_iou_intersection"] = int((band & gt["inner_band"]).sum())
            row["boundary_iou_union"] = int((band | gt["inner_band"]).sum())
            near = gt["distance"] <= self.cfg.primary_boundary_px
            row.update({"boundary_error_count": int(((pred != g) & near).sum()),
                        "boundary_band_pixels": int(near.sum()),
                        "interior_fn": int((~pred & g & ~near).sum()),
                        "interior_pixels": int((g & ~near).sum()),
                        "far_background_fp": int((pred & ~g & ~near).sum()),
                        "far_background_pixels": int((~g & ~near).sum())})
            pred_slices = ndi.find_objects(pl)
        components = []
        primary_pairs = matched[self.cfg.primary_iou]
        for gid in range(1, gt["n_gt"] + 1):
            area = int(gt["areas"][gid])
            size = self._size(area, cuts)
            hit = gid in primary_pairs
            border = gid in gt["border_ids"]
            row[f"{size}_gt"] += 1
            row[f"{size}_obj_tp"] += int(hit)
            row[f"{size}_positive_pixels"] += area
            row[f"{size}_tp_pixels"] += int(inside[gid])
            if not border:
                row[f"{size}_nonborder_gt"] += 1
                row[f"{size}_nonborder_obj_tp"] += int(hit)
            c = {"sample_id": gt["sample_id"], "gt_id": gid,
                 "component_key": f"{gt['sample_id']}::{gid}", "area_px": area,
                 "size_bin": size, "border_truncated": border,
                 "covered_pixels": int(inside[gid]), "coverage": float(inside[gid] / area),
                 "completely_missed": inside[gid] == 0,
                 "has_interior_at_primary_width": bool(area > band_pixels[gid]),
                 "matched_boundary_iou": np.nan, "matched_mask_iou": np.nan}
            for t, pairs in matched.items():
                c[f"detected_iou_{_tag(t)}"] = gid in pairs
                c[f"matched_iou_{_tag(t)}"] = pairs[gid][1] if gid in pairs else np.nan
            if hit:
                pid, iou = primary_pairs[gid]
                c["matched_mask_iou"] = iou
                if geometry:
                    gs, ps = gt["slices"][gid-1], pred_slices[pid-1]
                    region = tuple(slice(min(a.start, b.start), max(a.stop, b.stop))
                                   for a, b in zip(gs, ps))
                    gb = inner_band(gl[region] == gid, self.cfg.primary_boundary_px)
                    pb = inner_band(pl[region] == pid, self.cfg.primary_boundary_px)
                    c["matched_boundary_iou"] = safe_div(int((gb & pb).sum()), int((gb | pb).sum()))
            components.append(c)
        return row, components

    def _counts_to_metrics(self, sums: Mapping[str, float], n_pairs: int) -> dict:
        m = binary_metrics(sums)
        m.update({"gt_components": sums["n_gt"], "predicted_components": sums["n_pred"],
                  "pairs": n_pairs, "alerts_per_pair": safe_div(sums["n_pred"], n_pairs),
                  "completely_missed_fraction": safe_div(sums["completely_missed_gt"], sums["n_gt"])})
        for t in self.cfg.iou_thresholds:
            tag = _tag(t)
            tp, fp, fn = (sums[f"obj_{k}_{tag}"] for k in ("tp", "fp", "fn"))
            m.update({f"object_precision_{tag}": safe_div(tp, tp+fp),
                      f"object_recall_{tag}": safe_div(tp, tp+fn),
                      f"object_f1_{tag}": safe_div(2*tp, 2*tp+fp+fn),
                      f"false_alerts_per_pair_{tag}": safe_div(fp, n_pairs),
                      f"detected_components_{tag}": tp, f"missed_components_{tag}": fn})
        for t in self.cfg.coverage_thresholds:
            m[f"gt_coverage_recall_{_tag(t)}"] = safe_div(sums[f"coverage_hits_{_tag(t)}"], sums["n_gt"])
        for name in ("small", "medium", "large"):
            m[f"{name}_object_recall"] = safe_div(sums[f"{name}_obj_tp"], sums[f"{name}_gt"])
            m[f"{name}_pixel_recall"] = safe_div(sums[f"{name}_tp_pixels"], sums[f"{name}_positive_pixels"])
            m[f"{name}_nonborder_object_recall"] = safe_div(sums[f"{name}_nonborder_obj_tp"], sums[f"{name}_nonborder_gt"])
        if "boundary_pred_count" in sums:
            for r in self.cfg.boundary_tolerances_px:
                p = safe_div(sums[f"boundary_matched_pred_{r}"], sums["boundary_pred_count"])
                recall = safe_div(sums[f"boundary_matched_gt_{r}"], sums["boundary_gt_count"])
                if sums["boundary_pred_count"] == 0 and sums["boundary_gt_count"] > 0:
                    bf = 0.0
                elif sums["boundary_gt_count"] == 0 and sums["boundary_pred_count"] > 0:
                    bf = 0.0
                elif not np.isfinite(p+recall):
                    bf = np.nan
                else:
                    bf = 0.0 if p+recall == 0 else 2*p*recall/(p+recall)
                m[f"boundary_precision_{r}px"] = p
                m[f"boundary_recall_{r}px"] = recall
                m[f"boundary_f1_{r}px"] = bf
            m["boundary_iou"] = safe_div(sums["boundary_iou_intersection"], sums["boundary_iou_union"])
            m["boundary_error_rate"] = safe_div(sums["boundary_error_count"], sums["boundary_band_pixels"])
            m["interior_fn_rate"] = safe_div(sums["interior_fn"], sums["interior_pixels"])
            m["far_background_fp_rate"] = safe_div(sums["far_background_fp"], sums["far_background_pixels"])
        return m

    def _evaluate_operation(self, s: dict, split: str, threshold: float, cuts: list) -> tuple[pd.DataFrame, pd.DataFrame]:
        folder = self.cache / "analysis" / split / s["run_id"] / _threshold_key(threshold)
        marker = folder / "complete.json"
        if marker.exists():
            info = read_json(marker)
            if info["signature"] != self.signature or info["cutoffs"] != cuts:
                raise RuntimeError("Stale metric cache.")
            images = pd.read_csv(folder / "images.csv", dtype={"sample_id": str})
            objects = pd.read_csv(folder / "components.csv", dtype={"sample_id": str})
            return images, objects
        folder.mkdir(parents=True, exist_ok=True)
        image_rows, object_rows = [], []
        for i in range(len(self.datasets[split])):
            scores = self.score(s, split, i)
            row, components = self._evaluate_mask(scores >= np.float32(threshold), self.gt(split, i), cuts)
            image_rows.append(row); object_rows.extend(components)
        images = pd.DataFrame(image_rows)
        objects = pd.DataFrame(object_rows)
        if objects.empty:
            objects = pd.DataFrame(columns=["sample_id", "gt_id", "component_key", "area_px", "size_bin"])
        images.to_csv(folder / "images.csv", index=False)
        objects.to_csv(folder / "components.csv", index=False)
        write_json(marker, {"signature": self.signature, "cutoffs": cuts, "threshold": threshold})
        return images, objects

    def evaluate_split(self, split: str, plan: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        summaries, all_images, all_objects = [], [], []
        for s in self.specs:
            for policy, op in plan["operations"][s["run_id"]].items():
                if not op["attainable"]:
                    continue
                print(f"METRICS {split} {s['run_id']} {policy}", flush=True)
                images, objects = self._evaluate_operation(s, split, op["threshold"], plan["size_cutoffs_px"])
                identity = {"run_id": s["run_id"], "variant": s["variant"], "seed": s["seed"],
                            "primary": s["primary"], "policy": policy, "split": split,
                            "threshold": op["threshold"]}
                sums = images.select_dtypes(include=[np.number]).sum().to_dict()
                summaries.append({**identity, **self._counts_to_metrics(sums, len(images)),
                                  "target_pixel_recall": op.get("target_recall", np.nan),
                                  "validation_recall": op.get("validation_recall", np.nan)})
                all_images.append(images.assign(**identity))
                all_objects.append(objects.assign(**identity))
        summary, per_image, per_object = pd.DataFrame(summaries), pd.concat(all_images, ignore_index=True), pd.concat(all_objects, ignore_index=True)
        summary.to_csv(self.reports / f"{split}_summary.csv", index=False)
        per_image.to_csv(self.reports / f"{split}_per_image.csv", index=False)
        per_object.to_csv(self.reports / f"{split}_per_component.csv", index=False)
        return summary, per_image, per_object

    def benchmark(self, s: dict) -> dict:
        file = self.reports / "cost" / f"{s['run_id']}.json"
        if file.exists():
            r = read_json(file)
            if r["signature"] != self.signature:
                raise RuntimeError("Stale cost benchmark.")
            return r
        if self.cfg.cost_source == "saved_benchmark":
            old = read_json(self.root / "artifacts/final_models" / s["variant"] / "benchmark.json")
            r = {"signature": self.signature, "run_id": s["run_id"], "variant": s["variant"],
                 "latency_ms": old["latency_ms_per_pair"], "parameters": old["parameters"],
                 "peak_allocated_mib": old.get("peak_inference_vram_mb"),
                 "cost_source": "saved_benchmark", "comparable_protocol_verified": False,
                 "protocol": old.get("protocol", {}), "gpu": old.get("gpu_name")}
        else:
            self._release()
            baseline = torch.cuda.memory_allocated(self.device) if self.device.type == "cuda" else 0
            if self.device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(self.device)
            model = a = b = y = None
            try:
                model = self._load_model(s).to(self.device)
                sample = self.Dataset(self.dataset_root, "val", s["image_size"], transform=None)[0]
                a, b = (sample[k][None].to(self.device, dtype=torch.float32) for k in ("before", "after"))
                times = []
                with self.runtime(self.device), torch.inference_mode():
                    for _ in range(self.cfg.benchmark_warmup):
                        y = model(a, b); del y; y = None
                    for _ in range(self.cfg.benchmark_repetitions):
                        self._sync(); t0 = time.perf_counter()
                        y = model(a, b)
                        self._sync(); times.append(1000 * (time.perf_counter() - t0))
                        del y; y = None
                r = {"signature": self.signature, "run_id": s["run_id"], "variant": s["variant"],
                     "cost_source": "remeasure", "comparable_protocol_verified": True,
                     "latency_ms": float(np.mean(times)), "median_ms": float(np.median(times)),
                     "p95_ms": float(np.quantile(times, 0.95)), "samples_ms": times,
                     "parameters": sum(p.numel() for p in model.parameters()),
                     "peak_allocated_mib": ((torch.cuda.max_memory_allocated(self.device)-baseline)/2**20)
                         if self.device.type == "cuda" else None,
                     "gpu": torch.cuda.get_device_name(self.device) if self.device.type == "cuda" else None,
                     "protocol": {"precision": "fp32", "tf32": False, "batch_size": 1,
                         "warmup": self.cfg.benchmark_warmup, "repetitions": self.cfg.benchmark_repetitions,
                         "input_shape": list(a.shape), "cpu_threads": torch.get_num_threads(),
                         "torch": str(torch.__version__), "cuda": torch.version.cuda,
                         "scope": "synchronized forward-only wall latency, CPU launch included; inputs already on device",
                         "excluded": "I/O, decoding, resize, H2D, sigmoid, upsampling, thresholding, components, human review",
                         "interference": "Run on an otherwise idle GPU; utilization is not controlled by this notebook"}}
            finally:
                del model, a, b, y
                self._release()
        write_json(file, r)
        return r

    def _h3_pairs(self) -> list[dict]:
        pairs = []
        for a, b in itertools.combinations(self.primary, 2):
            if a["resolution"] == b["resolution"]:
                pairs.append({"a": a["run_id"], "b": b["run_id"], "kind": "cross_architecture_same_resolution"})
        if self.cfg.include_other_seeds:
            for a, b in itertools.combinations(self.specs, 2):
                if a["variant"] == b["variant"]:
                    pairs.append({"a": a["run_id"], "b": b["run_id"], "kind": "same_architecture_seed_control"})
        return pairs

    def _h3_plan(self, operations: dict, curves: dict) -> list[dict]:
        result = []
        for pair in self._h3_pairs():
            a, b = self.spec_by_id[pair["a"]], self.spec_by_id[pair["b"]]
            for target in self.cfg.recall_targets:
                policy = f"recall_{_tag(target)}"
                oa, ob = operations[a["run_id"]][policy], operations[b["run_id"]][policy]
                if not oa["attainable"] or not ob["attainable"]:
                    continue
                counts = []
                for i in range(len(self.datasets["val"])):
                    pa = self.score(a, "val", i) >= np.float32(oa["threshold"])
                    pb = self.score(b, "val", i) >= np.float32(ob["threshold"])
                    counts.append(pair_counts(pa, pb, self.gt("val", i)["mask"]))
                totals = pd.DataFrame(counts).sum().to_dict()
                r = safe_div(totals["tp_or"], totals["n_positive"])
                result.append({**pair, "policy": policy, "target": target,
                    "threshold_a": oa["threshold"], "threshold_b": ob["threshold"],
                    "validation_recall_a": oa["validation_recall"],
                    "validation_recall_b": ob["validation_recall"],
                    "validation_recall_gap": abs(oa["validation_recall"]-ob["validation_recall"]),
                    "validation_match_within_tolerance": abs(oa["validation_recall"]-ob["validation_recall"]) <= self.cfg.recall_match_tolerance,
                    "validation_or_recall": r,
                    "single_a_retarget": self._at_recall(curves[a["run_id"]], r),
                    "single_b_retarget": self._at_recall(curves[b["run_id"]], r)})
        return result

    def _paired_ci(self, frames: list[pd.DataFrame], function, split: str) -> dict:
        """Paired percentile bootstrap of aggregate counts, never IID pixels."""
        aligned = []
        ids = self.datasets[split].sample_ids
        for frame in frames:
            f = frame.set_index("sample_id").reindex(ids)
            if len(f) != len(ids) or f.index.has_duplicates:
                raise ValueError("Bootstrap inputs must have one row for every image pair.")
            if f.isna().all(axis=1).any():
                raise ValueError("Some image pairs are missing from paired bootstrap.")
            # Only numeric columns enter the resampled sums; sample IDs remain aligned.
            aligned.append(f.select_dtypes(include=[np.number]))
        labels = [self.cluster_map.get(f"{split}/{sid}", sid) for sid in ids]
        unique = sorted(set(labels))
        groups = [np.flatnonzero(np.array(labels) == c) for c in unique]
        totals = [f.sum().to_dict() for f in aligned]
        point = float(function(totals))
        rng = np.random.default_rng(self.cfg.bootstrap_seed)
        samples = []
        arrays = [np.stack([f.iloc[g].sum().to_numpy(float) for g in groups]) for f in aligned]
        names = [list(f.columns) for f in aligned]
        for _ in range(self.cfg.bootstrap_repetitions):
            draws = rng.integers(0, len(groups), size=len(groups))
            counts = [dict(zip(cols, array[draws].sum(axis=0))) for array, cols in zip(arrays, names)]
            val = float(function(counts))
            if math.isfinite(val):
                samples.append(val)
        return {"estimate": point, "ci_low": float(np.quantile(samples, 0.025)) if samples else np.nan,
                "ci_high": float(np.quantile(samples, 0.975)) if samples else np.nan,
                "valid_bootstrap_replicates": len(samples), "requested_replicates": self.cfg.bootstrap_repetitions,
                "resampling_units": len(groups), "resampling": "geographic_cluster" if self.cluster_map else "image_pair",
                "interval_type": "paired percentile 95%; conditional on checkpoints; unadjusted exploratory"}

    def h3(self, split: str, plan: dict) -> pd.DataFrame:
        rows, image_rows, baseline_rows = [], [], []
        tag = _tag(self.cfg.primary_iou)
        for p in plan["h3"]:
            a, b = self.spec_by_id[p["a"]], self.spec_by_id[p["b"]]
            print(f"H3 {split}: {p['a']} + {p['b']} {p['policy']}", flush=True)
            base = {k: p[k] for k in ("a", "b", "kind", "policy", "target", "threshold_a", "threshold_b",
                    "validation_recall_a", "validation_recall_b", "validation_recall_gap", "validation_match_within_tolerance", "validation_or_recall")}
            frame = []
            single = {"a": [], "b": []}
            for i in range(len(self.datasets[split])):
                gt = self.gt(split, i)
                sa, sb = self.score(a, split, i), self.score(b, split, i)
                ma, mb = sa >= np.float32(p["threshold_a"]), sb >= np.float32(p["threshold_b"])
                c = {"sample_id": gt["sample_id"], **pair_counts(ma, mb, gt["mask"])}
                # Recompute actual OR components: merging can change object matching.
                oa, ca = self._evaluate_mask(ma, gt, plan["size_cutoffs_px"], geometry=False)
                ob, cb = self._evaluate_mask(mb, gt, plan["size_cutoffs_px"], geometry=False)
                oo, _ = self._evaluate_mask(ma | mb, gt, plan["size_cutoffs_px"], geometry=False)
                aset = {x["gt_id"] for x in ca if x[f"detected_iou_{tag}"]}
                bset = {x["gt_id"] for x in cb if x[f"detected_iou_{tag}"]}
                c.update({"gt_components": gt["n_gt"], "detected_a": len(aset), "detected_b": len(bset),
                          "common_detected_gt": len(aset & bset), "set_union_detected_gt": len(aset | bset),
                          "actual_or_object_tp": oo[f"obj_tp_{tag}"], "actual_or_object_fp": oo[f"obj_fp_{tag}"],
                          "actual_or_pred_components": oo["n_pred"]})
                frame.append(c)
                for key, scores in (("a", sa), ("b", sb)):
                    op = p[f"single_{key}_retarget"]
                    if op["attainable"]:
                        r, _ = self._evaluate_mask(scores >= np.float32(op["threshold"]), gt, plan["size_cutoffs_px"], geometry=False)
                        single[key].append(r)
            df = pd.DataFrame(frame)
            total = df.select_dtypes(include=[np.number]).sum().to_dict()
            stats = complementarity(total)
            ci = self._paired_ci([df], lambda v: complementarity(v[0])["delta_recall"], split)
            best_obj = max(total["detected_a"], total["detected_b"])
            costs = {r["run_id"]: r for r in plan["costs"]}
            # Seed-control pairs use the same variant's primary architecture timing as a labelled proxy.
            def cost_of(s):
                primary = next(x for x in self.primary if x["variant"] == s["variant"])
                return costs[primary["run_id"]]["latency_ms"]
            result = {**base, "split": split, **total, **stats,
                "delta_recall_ci_low": ci["ci_low"], "delta_recall_ci_high": ci["ci_high"],
                "achieved_recall_gap": abs(stats["recall_a"]-stats["recall_b"]),
                "gt_detection_set_jaccard": safe_div(total["common_detected_gt"], total["set_union_detected_gt"]),
                "gt_detection_set_cg": safe_div(total["set_union_detected_gt"]-best_obj, total["gt_components"]-best_obj),
                "actual_or_object_recall": safe_div(total["actual_or_object_tp"], total["gt_components"]),
                "actual_or_object_precision": safe_div(total["actual_or_object_tp"], total["actual_or_object_tp"]+total["actual_or_object_fp"]),
                "sequential_forward_sum_ms_ESTIMATE": cost_of(a)+cost_of(b),
                "cost_note": "sum of single forward measurements, NOT measured ensemble latency; merge/postprocessing excluded"}
            rows.append(result)
            image_rows.append(df.assign(**base, split=split))
            for key in ("a", "b"):
                op = p[f"single_{key}_retarget"]
                if not single[key]:
                    baseline_rows.append({**base, "split": split, "single": key, "attainable_on_validation": False})
                    continue
                sf = pd.DataFrame(single[key])
                sm = self._counts_to_metrics(sf.select_dtypes(include=[np.number]).sum().to_dict(), len(sf))
                baseline_rows.append({**base, "split": split, "single": key, "attainable_on_validation": True,
                    "threshold_single_frozen": op["threshold"], "single_validation_recall": op["validation_recall"],
                    "single_forward_ms": cost_of(a if key == "a" else b), **sm,
                    "or_pixel_recall": stats["recall_or"], "or_pixel_precision": stats["precision_or"],
                    "or_actual_object_recall": result["actual_or_object_recall"],
                    "single_minus_or_pixel_recall": sm["pixel_recall"]-stats["recall_or"],
                    "single_minus_or_pixel_precision": sm["pixel_precision"]-stats["precision_or"]})
        df = pd.DataFrame(rows)
        df.to_csv(self.reports / f"H3_{split}_complementarity.csv", index=False)
        pd.DataFrame(baseline_rows).to_csv(self.reports / f"H3_{split}_OR_vs_retuned_single.csv", index=False)
        if image_rows:
            pd.concat(image_rows, ignore_index=True).to_csv(self.reports / f"H3_{split}_per_image.csv", index=False)
        return df

    def _frame(self, images: pd.DataFrame, run_id: str, policy: str = "val_f1") -> pd.DataFrame:
        f = images.loc[(images["run_id"] == run_id) & (images["policy"] == policy)].copy()
        if f.empty:
            raise ValueError(f"No metrics for {run_id}/{policy}")
        return f

    def h1(self, split: str, images: pd.DataFrame) -> pd.DataFrame:
        size_rows, contrasts = [], []
        for (rid, policy), group in images.groupby(["run_id", "policy"], sort=False):
            s = self.spec_by_id[rid]
            total = group.select_dtypes(include=[np.number]).sum()
            for size in ("small", "medium", "large"):
                size_rows.append({"split": split, "run_id": rid, "variant": s["variant"], "seed": s["seed"],
                    "primary": s["primary"], "policy": policy, "size_bin": size,
                    "gt_count": total[f"{size}_gt"], "detected_count": total[f"{size}_obj_tp"],
                    "positive_pixels": total[f"{size}_positive_pixels"], "true_positive_pixels": total[f"{size}_tp_pixels"],
                    "object_recall": safe_div(total[f"{size}_obj_tp"], total[f"{size}_gt"]),
                    "pixel_recall": safe_div(total[f"{size}_tp_pixels"], total[f"{size}_positive_pixels"]),
                    "nonborder_gt_count": total[f"{size}_nonborder_gt"],
                    "nonborder_object_recall": safe_div(total[f"{size}_nonborder_obj_tp"], total[f"{size}_nonborder_gt"])})
        pairs = []
        for family in ("light", "medium", "heavy"):
            fs = [s for s in self.specs if s["family"] == family and s["resolution"] == "full"]
            rs = [s for s in self.specs if s["family"] == family and s["resolution"] == "resized"]
            if not fs or not rs:
                continue
            pairs.append((next(s for s in fs if s["primary"]), next(s for s in rs if s["primary"]), "deployment_pair"))
            if self.cfg.include_other_seeds:
                for a in fs:
                    for b in rs:
                        if a["seed"] == b["seed"]:
                            pairs.append((a, b, "same_seed_sensitivity"))
        for a, b, kind in pairs:
            fa, fb = self._frame(images, a["run_id"]), self._frame(images, b["run_id"])
            for metric, numerator, denominator in (("object_recall", "obj_tp", "gt"),
                                                   ("pixel_recall", "tp_pixels", "positive_pixels")):
                def small_large(v, num=numerator, den=denominator):
                    diff = lambda size: safe_div(v[0][f"{size}_{num}"], v[0][f"{size}_{den}"]) - safe_div(v[1][f"{size}_{num}"], v[1][f"{size}_{den}"])
                    return diff("small") - diff("large")
                contrasts.append({"split": split, "family": a["family"], "full": a["run_id"],
                    "resized": b["run_id"], "comparison": kind, "metric": metric,
                    "contrast": "(Full-Resized)_small - (Full-Resized)_large; positive supports size-dependent advantage",
                    "same_architecture_parameters": a["model_parameters"] == b["model_parameters"],
                    **self._paired_ci([fa, fb], small_large, split)})
        pd.DataFrame(size_rows).to_csv(self.reports / f"H1_{split}_size_metrics.csv", index=False)
        df = pd.DataFrame(contrasts)
        df.to_csv(self.reports / f"H1_{split}_paired_contrasts.csv", index=False)
        return df

    def h2(self, split: str, images: pd.DataFrame, objects: pd.DataFrame) -> pd.DataFrame:
        rows, common_rows = [], []
        tag = _tag(self.cfg.primary_iou)
        r = self.cfg.primary_boundary_px
        for h in [s for s in self.primary if s["family"] == "heavy"]:
            for base in [s for s in self.primary if s["resolution"] == h["resolution"] and s["family"] != "heavy"]:
                fh, fb = self._frame(images, h["run_id"]), self._frame(images, base["run_id"])
                metrics = ["pixel_f1", f"object_recall_{tag}", f"boundary_f1_{r}px", "boundary_iou",
                           "boundary_error_rate", "interior_fn_rate", "far_background_fp_rate", "gt_coverage_recall_050"]
                for metric in metrics:
                    if metric == "gt_coverage_recall_050" and 0.5 not in self.cfg.coverage_thresholds:
                        continue
                    def difference(v, name=metric):
                        return self._counts_to_metrics(v[0], len(fh))[name] - self._counts_to_metrics(v[1], len(fb))[name]
                    rows.append({"split": split, "heavy": h["run_id"], "baseline": base["run_id"],
                                 "metric": metric, "direction": "Heavy minus baseline",
                                 **self._paired_ci([fh, fb], difference, split)})
                oh = objects.loc[(objects["run_id"] == h["run_id"]) & (objects["policy"] == "val_f1")]
                ob = objects.loc[(objects["run_id"] == base["run_id"]) & (objects["policy"] == "val_f1")]
                merged = oh.merge(ob, on=["sample_id", "gt_id"], suffixes=("_heavy", "_base"), validate="one_to_one")
                if len(merged) != len(oh) or len(merged) != len(ob):
                    raise ValueError("GT component IDs do not align across models.")
                dh, db = merged[f"detected_iou_{tag}_heavy"].astype(bool), merged[f"detected_iou_{tag}_base"].astype(bool)
                common = merged.loc[dh & db].copy()
                common.to_csv(self.reports / f"H2_{split}_{h['variant']}_vs_{base['variant']}_common_components.csv", index=False)
                ch, cb = fh.select_dtypes(include=[np.number]).sum(), fb.select_dtypes(include=[np.number]).sum()
                total_reduction = (cb["fp"]+cb["fn"]) - (ch["fp"]+ch["fn"])
                near_reduction = cb["boundary_error_count"] - ch["boundary_error_count"]
                for metric in ("matched_mask_iou", "matched_boundary_iou"):
                    conditional = []
                    for sid in self.datasets[split].sample_ids:
                        group = common.loc[common["sample_id"] == sid]
                        av, bv = group[f"{metric}_heavy"], group[f"{metric}_base"]
                        valid = av.notna() & bv.notna()
                        conditional.append({"sample_id": sid, "sum_heavy": av[valid].sum(),
                                            "sum_base": bv[valid].sum(), "n_common": int(valid.sum())})
                    cf = pd.DataFrame(conditional)
                    ci = self._paired_ci([cf], lambda v: safe_div(v[0]["sum_heavy"]-v[0]["sum_base"], v[0]["n_common"]), split)
                    common_rows.append({"split": split, "heavy": h["run_id"], "baseline": base["run_id"],
                        "metric": metric, "conditioning": "GT components matched by BOTH at primary IoU; not all objects",
                        "common_gt": int((dh & db).sum()), "heavy_only_gt": int((dh & ~db).sum()),
                        "baseline_only_gt": int((~dh & db).sum()), "missed_by_both": int((~dh & ~db).sum()),
                        "total_pixel_error_reduction": total_reduction,
                        "boundary_band_error_reduction": near_reduction,
                        "boundary_share_of_net_error_reduction": safe_div(near_reduction, total_reduction),
                        "share_note": "error-count decomposition, NOT a causal decomposition of F1; can exceed 1 if other errors increase",
                        **ci})
        df = pd.DataFrame(rows)
        df.to_csv(self.reports / f"H2_{split}_geometry_contrasts.csv", index=False)
        pd.DataFrame(common_rows).to_csv(self.reports / f"H2_{split}_common_detection_geometry.csv", index=False)
        return df

    def h4(self, split: str, summary: pd.DataFrame, images: pd.DataFrame) -> dict:
        selected = summary.loc[summary["primary"] & (summary["policy"] == "val_f1")].copy()
        tag = _tag(self.cfg.primary_iou)
        selected["pixel_f1_rank"] = selected["pixel_f1"].rank(method="min", ascending=False)
        selected["object_recall_rank"] = selected[f"object_recall_{tag}"].rank(method="min", ascending=False)
        for t in self.cfg.iou_thresholds:
            selected[f"object_rank_iou_{_tag(t)}"] = selected[f"object_recall_{_tag(t)}"].rank(method="min", ascending=False)
        selected.to_csv(self.reports / f"H4_{split}_rankings.csv", index=False)
        pairs = []
        for a, b in itertools.combinations(self.primary, 2):
            fa, fb = self._frame(images, a["run_id"]), self._frame(images, b["run_id"])
            cf1 = self._paired_ci([fa, fb], lambda v: binary_metrics(v[0])["pixel_f1"]-binary_metrics(v[1])["pixel_f1"], split)
            cr = self._paired_ci([fa, fb], lambda v: safe_div(v[0][f"obj_tp_{tag}"], v[0]["n_gt"])-safe_div(v[1][f"obj_tp_{tag}"], v[1]["n_gt"]), split)
            pairs.append({"split": split, "a": a["run_id"], "b": b["run_id"],
                "delta_pixel_f1": cf1["estimate"], "f1_ci_low": cf1["ci_low"], "f1_ci_high": cf1["ci_high"],
                "delta_object_recall": cr["estimate"], "recall_ci_low": cr["ci_low"], "recall_ci_high": cr["ci_high"],
                "observed_rank_inversion": cf1["estimate"]*cr["estimate"] < 0,
                "opposite_sign_intervals_excluding_zero": (cf1["ci_low"] > 0 and cr["ci_high"] < 0) or (cf1["ci_high"] < 0 and cr["ci_low"] > 0),
                "interval_note": "exploratory, unadjusted multiple pairwise comparisons"})
        pd.DataFrame(pairs).to_csv(self.reports / f"H4_{split}_pairwise_inversions.csv", index=False)
        # Selection is used ONLY when called on validation, then frozen for test.
        pixel_winner = selected.sort_values(["pixel_f1", "variant"], ascending=[False, True]).iloc[0]["run_id"]
        obj_winner = selected.sort_values([f"object_recall_{tag}", "pixel_f1", "variant"], ascending=[False, False, True]).iloc[0]["run_id"]
        ranks = selected[["pixel_f1_rank", "object_recall_rank"]]
        correlation = ranks.corr(method="pearson").iloc[0, 1]  # Rank correlation; six pipelines only.
        result = {"split": split, "max_pixel_f1_run": pixel_winner,
                  "max_object_recall_run": obj_winner,
                  "different_winners": pixel_winner != obj_winner,
                  "rank_correlation_descriptive": correlation,
                  "warning": "Same winner does not establish general metric equivalence. Test rankings are descriptive, not a deployment-selection rule."}
        write_json(self.reports / f"H4_{split}_selection_and_rank_summary.json", result)
        return result

    def h5(self, split: str, summary: pd.DataFrame, costs: list[dict]) -> dict:
        cost_frame = pd.DataFrame([{k: r.get(k) for k in ("run_id", "latency_ms", "parameters", "peak_allocated_mib",
                                                        "cost_source", "comparable_protocol_verified", "gpu")} for r in costs])
        selected = summary.loc[summary["primary"]].merge(cost_frame, on="run_id", validate="many_to_one")
        tag = _tag(self.cfg.primary_iou)
        primary = selected.loc[selected["policy"] == "val_f1"].copy()
        def enrich(df):
            df = df.copy()
            df["pareto_pixel_f1_latency"] = pareto_flags(df, ["pixel_f1"], ["latency_ms"])
            df["pareto_recall_false_alerts_latency"] = pareto_flags(df, [f"object_recall_{tag}"], [f"false_alerts_per_pair_{tag}", "latency_ms"])
            # Occupied device-time proxy under serial batch=1/exclusive hourly billing.
            hours = self.cfg.scenario_pairs * df["latency_ms"] / 3_600_000
            df["forward_hours_proxy_per_scenario"] = hours
            df["scenario_pairs"] = self.cfg.scenario_pairs
            df["scenario_compute_cost"] = hours * self.cfg.gpu_price_per_hour if self.cfg.gpu_price_per_hour is not None and self.device.type == "cuda" else np.nan
            alerts = df["alerts_per_pair"] * self.cfg.scenario_pairs
            misses = df[f"missed_components_{tag}"] / df["pairs"] * self.cfg.scenario_pairs
            df["scenario_review_cost"] = alerts*self.cfg.review_price_per_alert if self.cfg.review_price_per_alert is not None else np.nan
            df["scenario_missed_component_loss"] = misses*self.cfg.missed_component_price if self.cfg.missed_component_price is not None else np.nan
            df["scenario_total_cost"] = df[["scenario_compute_cost", "scenario_review_cost", "scenario_missed_component_loss"]].sum(axis=1, min_count=3)
            df["cost_scope"] = "scenario proxy; forward-only compute + every predicted component reviewed + assumed missed-component loss; not observed end-to-end cost"
            return df
        primary = enrich(primary)
        all_points = enrich(selected.drop_duplicates(["run_id", "threshold"]))
        primary.to_csv(self.reports / f"H5_{split}_quality_cost_pareto.csv", index=False)
        all_points.to_csv(self.reports / f"H5_{split}_all_operating_points.csv", index=False)
        comparisons = []
        for a, b in itertools.permutations(primary.to_dict("records"), 2):
            n = a["pairs"]
            if n != b["pairs"]:
                raise ValueError("Models have different evaluation populations.")
            extra = a[f"detected_components_{tag}"] - b[f"detected_components_{tag}"]
            ds = n * (a["latency_ms"]-b["latency_ms"])/1000
            dalerts = n*(a["alerts_per_pair"]-b["alerts_per_pair"])
            dc = ds/3600*self.cfg.gpu_price_per_hour if self.cfg.gpu_price_per_hour is not None and self.device.type == "cuda" else np.nan
            dr = dalerts*self.cfg.review_price_per_alert if self.cfg.review_price_per_alert is not None else np.nan
            comparisons.append({"split": split, "candidate": a["run_id"], "baseline": b["run_id"],
                "extra_detected_components_NET": extra, "extra_review_alerts": dalerts,
                "extra_false_alerts": n*(a[f"false_alerts_per_pair_{tag}"]-b[f"false_alerts_per_pair_{tag}"]),
                "extra_forward_seconds_on_this_split": ds,
                "extra_seconds_per_net_additional_detected_component": safe_div(ds, extra) if ds > 0 else np.nan,
                "break_even_missed_component_value": safe_div(dc+dr, extra),
                "equation": "candidate cheaper iff extra_compute_cost + extra_review_cost < missed_component_value * net_additional_detections",
                "warning": "Net count gain is not unique-new-objects count; model overlap is analysed in H3. No universal utility threshold."})
        pd.DataFrame(comparisons).to_csv(self.reports / f"H5_{split}_marginal_and_break_even.csv", index=False)
        constraints_used = any(x is not None for x in (self.cfg.minimum_object_recall,
                self.cfg.maximum_false_alerts_per_pair, self.cfg.maximum_forward_ms))
        feasible = all_points.copy()
        if self.cfg.minimum_object_recall is not None:
            feasible = feasible.loc[feasible[f"object_recall_{tag}"] >= self.cfg.minimum_object_recall]
        if self.cfg.maximum_false_alerts_per_pair is not None:
            feasible = feasible.loc[feasible[f"false_alerts_per_pair_{tag}"] <= self.cfg.maximum_false_alerts_per_pair]
        if self.cfg.maximum_forward_ms is not None:
            feasible = feasible.loc[feasible["latency_ms"] <= self.cfg.maximum_forward_ms]
        chosen = None
        if constraints_used and not feasible.empty:
            chosen = feasible.sort_values(["latency_ms", f"false_alerts_per_pair_{tag}", "pixel_f1", "variant"],
                                          ascending=[True, True, False, True]).iloc[0]
            chosen = {"run_id": chosen["run_id"], "policy": chosen["policy"], "threshold": chosen["threshold"]}
        cost_winner = None
        if all_points["scenario_total_cost"].notna().any():
            winner = all_points.sort_values("scenario_total_cost").iloc[0]
            cost_winner = {"run_id": winner["run_id"], "policy": winner["policy"], "threshold": winner["threshold"]}
        result = {"split": split, "user_constraints_supplied": constraints_used,
                  "fastest_feasible": chosen, "scenario_cost_winner": cost_winner,
                  "point_estimate_pareto_only": True,
                  "warning": "No cost/practical-utility claim without explicit workload and error costs. Test does not re-select deployment."}
        write_json(self.reports / f"H5_{split}_scenario_selection.json", result)
        return result

    def freeze_validation(self) -> dict:
        """Every selectable setting uses validation; existing frozen plans are immutable."""
        if self.plan is not None:
            if self.plan["signature"] != self.signature:
                raise RuntimeError("Frozen protocol belongs to different inputs.")
            print("Existing frozen protocol loaded; no validation retuning.")
            return self.plan
        self._reproduction()
        curves = {s["run_id"]: self._curve(s) for s in self.specs}
        pd.concat(curves.values(), ignore_index=True).to_csv(self.reports / "validation_PR_grid.csv", index=False)
        plan = {"signature": self.signature, "size_cutoffs_px": self._size_cutoffs(),
                "size_bin_rule": "small <= q1; q1 < medium <= q2; large > q2; original GT area",
                "operations": self._make_operations(curves), "costs": []}
        print("Validation thresholds and size bins chosen; evaluating validation masks...", flush=True)
        summary, images, objects = self.evaluate_split("val", plan)
        for s in self.primary:
            print(f"COST {s['run_id']}: {self.cfg.cost_source}", flush=True)
            plan["costs"].append(self.benchmark(s))
        plan["h3"] = self._h3_plan(plan["operations"], curves)
        self.h1("val", images)
        self.h2("val", images, objects)
        self.h3("val", plan)
        plan["h4_validation_selection"] = self.h4("val", summary, images)
        plan["h5_validation_selection"] = self.h5("val", summary, plan["costs"])
        plan["frozen_at_utc"] = datetime.now(timezone.utc).isoformat()
        plan["test_access_policy"] = "No retuning after freeze. Test has been used previously in Research I; post-hoc evaluation, not new preregistered evidence."
        write_json(self.out / "protocol_frozen.json", plan)
        self.plan = _json_value(plan)
        print(f"FROZEN: {self.out / 'protocol_frozen.json'}")
        return self.plan

    def analyze_test(self) -> pd.DataFrame:
        if self.plan is None:
            raise RuntimeError("Run freeze_validation() first.")
        for s in self.specs:
            p = self.cache / s["run_id"] / "test/complete.json"
            if not p.exists() or read_json(p)["signature"] != self.signature:
                raise RuntimeError(f"Test inference not complete: {s['run_id']}")
        summary, images, objects = self.evaluate_split("test", self.plan)
        self.h1("test", images)
        self.h2("test", images, objects)
        self.h3("test", self.plan)
        self.h4("test", summary, images)
        self.h5("test", summary, self.plan["costs"])
        # Explicitly report test outcomes of choices made on validation.
        selected_rows = []
        for reason, choice in (
            ("max_validation_pixel_f1", {"run_id": self.plan["h4_validation_selection"]["max_pixel_f1_run"], "policy": "val_f1"}),
            ("max_validation_object_recall", {"run_id": self.plan["h4_validation_selection"]["max_object_recall_run"], "policy": "val_f1"}),
            ("fastest_feasible_on_validation", self.plan["h5_validation_selection"]["fastest_feasible"]),
            ("minimum_scenario_cost_on_validation", self.plan["h5_validation_selection"]["scenario_cost_winner"])):
            if choice is not None:
                record = summary.loc[(summary["run_id"] == choice["run_id"]) & (summary["policy"] == choice["policy"])].copy()
                selected_rows.append(record.assign(selection_reason=reason))
        if selected_rows:
            pd.concat(selected_rows, ignore_index=True).to_csv(self.reports / "validation_selected_test_outcomes.csv", index=False)
        self._seed_summary(summary)
        self._write_interpretation_guide()
        write_json(self.reports / "COMPLETE.json", {"signature": self.signature,
            "completed_utc": datetime.now(timezone.utc).isoformat(), "hypotheses_proved": None,
            "note": "Measurements completed; interpretation is separate. No training was performed."})
        return summary

    def _seed_summary(self, summary: pd.DataFrame) -> None:
        m = summary.loc[summary["policy"] == "val_f1"]
        metrics = ["pixel_f1", "pixel_recall", f"object_recall_{_tag(self.cfg.primary_iou)}", "boundary_iou"]
        rows = []
        for variant, g in m.groupby("variant"):
            for metric in metrics:
                rows.append({"variant": variant, "metric": metric, "n_seeds_evaluated": len(g),
                    "seeds": ",".join(str(x) for x in g["seed"]), "mean": g[metric].mean(),
                    "sample_sd_ddof1": g[metric].std(ddof=1) if len(g) > 1 else np.nan,
                    "scope": "available checkpoints; deployment is counted once; unequal seed sets not a balanced architecture experiment"})
        pd.DataFrame(rows).to_csv(self.reports / "test_seed_sensitivity.csv", index=False)

    def _write_interpretation_guide(self) -> None:
        text = """# Reading the H1--H5 evidence

No hypothesis is automatically declared proved. These are exploratory analyses
of already-trained pipelines and a previously inspected test split.

- H1: `H1_test_paired_contrasts.csv` estimates (Full-Resized)_small minus
  (Full-Resized)_large. Read size-group numerators/denominators as well as CIs.
  Light/Medium have different widths between Full and Resized. A positive
  contrast is not proof of a pure causal resolution effect.
- H2: inspect pixel F1, object recall, boundary/interior errors and geometry on
  COMMON detected GT components. Error-count share is not an F1 decomposition.
  Matching at IoU>=0.5 is itself geometry-sensitive. Use IoU sensitivity,
  GT coverage and completely-missed counts as complementary checks.
- H3: TP/FP/mask Jaccard are distinct. CG is this study's operational
  recovery fraction, undefined if the better model already has recall 1.
  Report its numerator/denominator and delta recall. Matching validation
  recall does not force matching test recall. Compare OR to single models
  whose thresholds were retargeted on validation, not on test. GT detection-set
  union is not the actual OR-mask detection result; merged masks can lose matches.
- H4: inspect ranking reversals and paired CIs. Test rankings are descriptive.
  For selection, use `validation_selected_test_outcomes.csv`. Same winner in
  this sample does not establish universal equivalence of pixel F1/object recall.
- H5: Pareto labels use point estimates. Forward latency excludes preprocessing,
  connected components, I/O, human work and semantic-stage inference. Monetary
  figures are only user scenarios; missing rates remain missing, never zero.
  Operator review covers ALL predicted components, not just false positives.
  Report inequalities/break-even thresholds rather than a universal best model.

GT connected components are NOT guaranteed building instances. Border-touching
regions are retained and flagged; size-group nonborder sensitivity is exported.
Small objects without an eroded interior are reported, not silently discarded.
Bootstrap resamples image pairs (or supplied geographic clusters); it does not
measure training variance. Spatially related images can violate the default
independence assumption. Intervals are exploratory and not multiplicity-adjusted.
One available seed cannot support a training-stability conclusion.
A LEVIR-CD result does not establish performance on SECOND or the full WHERE->WHAT system.
"""
        (self.reports / "INTERPRETATION.md").write_text(text, encoding="utf-8")

    def plot_reports(self, split: str = "test", show: bool = True) -> list[Path]:
        """Separate matplotlib figures, default colors, no seaborn or subplots."""
        import matplotlib.pyplot as plt
        directory = self.out / "figures"
        directory.mkdir(exist_ok=True)
        files = []
        def finish(fig, name):
            path = directory / name
            fig.tight_layout(); fig.savefig(path, dpi=160)
            files.append(path)
            if show:
                plt.show()
            plt.close(fig)
        curves = pd.read_csv(self.reports / "validation_PR_grid.csv")
        fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111)
        for s in self.primary:
            c = curves.loc[curves["run_id"] == s["run_id"]].sort_values("threshold", ascending=False)
            ax.plot(c["pixel_recall"], c["pixel_precision"], label=s["variant"])
        ax.set(xlabel="Pixel recall (validation, native GT grid)", ylabel="Pixel precision", title="Validation threshold curves")
        ax.legend(); finish(fig, "validation_PR.png")
        sizes = pd.read_csv(self.reports / f"H1_{split}_size_metrics.csv")
        fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111)
        for s in self.primary:
            d = sizes.loc[(sizes["run_id"] == s["run_id"]) & (sizes["policy"] == "val_f1")].set_index("size_bin").reindex(["small", "medium", "large"])
            ax.plot(["small", "medium", "large"], d["object_recall"], marker="o", label=s["variant"])
        ax.set(xlabel="GT size bin fixed on validation", ylabel=f"Component recall, IoU >= {self.cfg.primary_iou}", title=f"H1 — {split}")
        ax.legend(); finish(fig, f"H1_{split}_size_recall.png")
        d = pd.read_csv(self.reports / f"H4_{split}_rankings.csv")
        fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111)
        ax.scatter(d["pixel_f1"], d[f"object_recall_{_tag(self.cfg.primary_iou)}"])
        for r in d.itertuples():
            ax.annotate(r.variant, (r.pixel_f1, getattr(r, f"object_recall_{_tag(self.cfg.primary_iou)}")), xytext=(4, 4), textcoords="offset points")
        ax.set(xlabel="Pixel F1", ylabel="Component recall", title=f"H4 — {split}; thresholds selected on validation")
        finish(fig, f"H4_{split}_pixel_vs_object.png")
        d = pd.read_csv(self.reports / f"H5_{split}_quality_cost_pareto.csv")
        fig = plt.figure(figsize=(8, 5)); ax = fig.add_subplot(111)
        ax.scatter(d["latency_ms"], d[f"object_recall_{_tag(self.cfg.primary_iou)}"])
        for r in d.itertuples():
            ax.annotate(r.variant, (r.latency_ms, getattr(r, f"object_recall_{_tag(self.cfg.primary_iou)}")), xytext=(4, 4), textcoords="offset points")
        ax.set(xlabel="Forward-only latency, ms/pair", ylabel="Component recall", title=f"H5 — {split}; batch=1 FP32")
        finish(fig, f"H5_{split}_recall_vs_forward_cost.png")
        return files

    def export_results(self) -> Path:
        self._write_interpretation_guide()
        target = self.out / f"{self.cfg.run_name}_results.zip"
        temp = target.with_suffix(".tmp")
        allowed = [self.out / "run_manifest.json", self.out / "protocol_frozen.json"]
        allowed += list(self.reports.rglob("*"))
        allowed += list((self.out / "figures").rglob("*")) if (self.out / "figures").exists() else []
        with ZipFile(temp, "w", ZIP_DEFLATED) as z:
            for p in sorted(set(allowed)):
                if p.is_file():
                    z.write(p, p.relative_to(self.out))
        temp.replace(target)
        print(f"Results archive (no weights/images/score cache): {target}")
        return target


def run_self_tests() -> dict:
    """Synthetic metric checks only; no project, checkpoint or dataset needed."""
    z = np.zeros((24, 24), bool)
    gt = z.copy(); gt[2:6, 2:6] = True; gt[12:16, 12:16] = True
    same = metric_counts(gt, gt)
    assert binary_metrics(same)["pixel_f1"] == 1
    assert math.isnan(binary_metrics(metric_counts(z, z))["pixel_f1"])
    a = z.copy(); a[2:6, 2:6] = True
    b = z.copy(); b[12:16, 12:16] = True
    c = complementarity(pair_counts(a, b, gt))
    assert c["complementarity_gain"] == 1 and c["delta_recall"] == .5
    assert complementarity(pair_counts(a, a, gt))["complementarity_gain"] == 0
    assert math.isnan(complementarity(pair_counts(gt, gt, gt))["complementarity_gain"])
    l, n = ndi.label(gt, structure=np.ones((3, 3)))
    area = np.bincount(l.ravel())
    matches, _ = match_components(l, l, area, area, (.1, .25, .5))
    assert all(len(v) == n == 2 for v in matches.values())
    gl = np.array([[1, 1, 0, 2, 2]])
    pl = np.array([[1, 1, 1, 1, 1]])
    matches, _ = match_components(gl, pl, np.bincount(gl.ravel()), np.bincount(pl.ravel()), (.1, .5))
    assert len(matches[.1]) == 1 and len(matches[.5]) == 0  # merged predictions don't match two GTs.
    grid = np.array([0, .1, .5, 1], dtype=np.float32)
    scores = np.array([[0, .1, .5, 1], [.05, .4, .9, 1]], dtype=np.float32)
    truth = np.array([[1, 0, 1, 0], [1, 0, 1, 0]], dtype=bool)
    tp, fp = exact_threshold_counts(scores, truth, grid)
    for i, t in enumerate(grid):
        expected = metric_counts(scores >= t, truth)
        assert tp[i] == expected["tp"] and fp[i] == expected["fp"]
    p = pd.DataFrame({"quality": [.8, .7, .9], "cost": [1, 2, 3]})
    assert pareto_flags(p, ["quality"], ["cost"]).tolist() == [True, False, True]
    assert np.all(boundary(gt) <= gt)
    return {"synthetic_metric_tests": "PASSED", "real_checkpoint_inference_tested": False}
'''

import hashlib
IMPLEMENTATION_SHA256 = hashlib.sha256(CORE_SOURCE.encode('utf-8')).hexdigest()
exec(compile(CORE_SOURCE, 'levir_hypotheses_core.embedded.py', 'exec'), globals())
print('Analysis implementation:', IMPLEMENTATION_VERSION, IMPLEMENTATION_SHA256[:12])

Analysis implementation: levir-H1-H5-1.0.0 47f1d7aa65e2


## 2. Verify project, provenance and existing ROI implementation

In [3]:
import sys, os, json, zipfile, hashlib, time, warnings
from datetime import datetime, timezone
from collections import defaultdict
import numpy as np
import pandas as pd
import cv2
import torch
from scipy import ndimage as ndi
from IPython.display import display, FileLink
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PROJECT_ROOT=PROJECT_ROOT.expanduser().resolve()
SRC=PROJECT_ROOT/'src'
if not SRC.is_dir():
    raise FileNotFoundError(f'Missing project src/: {SRC}')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from satchange.semantic.roi import extract_rois, ROISettings
import satchange.semantic.roi as roi_module
if not Path(roi_module.__file__).resolve().is_relative_to(SRC):
    raise RuntimeError('ROI imported from unexpected location: '+str(roi_module.__file__))

v1_root = PROJECT_ROOT/'artifacts/hypothesis_tests'/V1_RUN
v1_manifest=read_json(v1_root/'run_manifest.json')
core_sha=hashlib.sha256(CORE_SOURCE.encode('utf-8')).hexdigest()
if core_sha != v1_manifest['protocol']['implementation_sha256']:
    raise RuntimeError('H10 original core differs from the H1–H5 protocol. Do not bypass this guard.')
exec(compile(CORE_SOURCE,'unchanged_original_06.py','exec'),globals())
settings=Settings(**v1_manifest['protocol']['settings'])
if Path(settings.project_root).resolve()!=PROJECT_ROOT:
    raise RuntimeError('Project root differs from H1–H5 source project.')
experiment=Experiment(settings)
if experiment.signature!=v1_manifest['signature']:
    raise RuntimeError('H1–H5 signature mismatch: checkpoints, cache or datasets changed.')
variant_specs={s['variant']:s for s in experiment.primary}
if not set(ROI_MODELS+TOPOLOGY_MODELS).issubset(variant_specs):
    raise ValueError('Some requested model variants are missing from original six checkpoints.')
ns=len(experiment.datasets['val'])
if ns<16:raise RuntimeError('Expected complete original validation dataset.')
for variant in set(ROI_MODELS+TOPOLOGY_MODELS):
    s=variant_specs[variant]
    missing=[j for j in range(ns) if not experiment._native_path(s,'val',j).is_file()]
    if missing:raise FileNotFoundError(f'Incomplete prediction cache for {variant}: {missing[:5]}')
OUTPUT=PROJECT_ROOT/'artifacts/hypothesis_tests'/OUT_NAME
OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/'figures').mkdir(exist_ok=True)
roi_sha=hashlib.sha256(Path(roi_module.__file__).read_bytes()).hexdigest()
sel_rng=np.random.default_rng(INTERNAL_AUDIT_SEED)
perm=sel_rng.permutation(ns)
dev_indices=set(perm[:ns//2].tolist())
audit_indices=set(perm[ns//2:].tolist())
assert len(dev_indices)+len(audit_indices)==ns and not (dev_indices&audit_indices)
pair_split={i:('dev' if i in dev_indices else 'audit') for i in range(ns)}
print('Original score / GT manifest verified:',experiment.signature[:18], '| core:',core_sha[:18])
print('Project ROI function:',roi_module.__file__,'| sha256:',roi_sha[:16])
print('Validation pairs:',ns,'| internal dev:',len(dev_indices),'| audit:',len(audit_indices))
try:
    from skimage.segmentation import watershed
    from skimage.feature import peak_local_max
    WATERSHED_AVAILABLE=True
except ImportError:
    WATERSHED_AVAILABLE=False
    if RUN_WATERSHED:
        print('OPTIONAL: scikit-image absent; distance-watershed experiment will be skipped. No package installation.')

Hashing validation/test inputs for cache integrity (no test metrics)...
NOTE: light: Full and Resized have different architecture parameters. H1 is a pipeline comparison, not a pure resolution ablation.
NOTE: medium: Full and Resized have different architecture parameters. H1 is a pipeline comparison, not a pure resolution ablation.
NOTE: heavy_resized: only one seed file; no estimate of training-seed variance.
NOTE: final_models_metrics.csv mixes validation/test fields. It is NOT used for evaluation or model ranking.
NOTE: Saved cost CSV and per-model benchmark JSON are separate measurements; do not mix their protocols. H5 defaults to one fresh common forward-only benchmark.
NOTE: Connected GT components are change regions, NOT guaranteed building-instance labels.
NOTE: Earlier test scores were already inspected: these analyses are exploratory/post-hoc.
NOTE: A deployment checkpoint is not an independent replicate of its source seed.
NOTE: No automatic hypothesis-proved labels are pro

## 3. ROI strategies — original project extractor as the only component-to-crop implementation

**All methods use exactly the same predicted binary mask and native image geometry for a given `(model, threshold, image)`**. Only crop selection/geometry changes.

- `project_128`: unchanged project component crop extractor, native minimum crop size 128.
- `project_64`: same production extractor, minimum crop size 64 (pure parameter change).
- `merge_overlaps`: union of *overlapping production crop boxes* if overlap is large enough and merged area stays below a safety cap. This is an experiment-only box post-processor, not a new model or reimplementation of component extraction.
- `prune_redundant`: greedy elimination of crops whose predicted positive pixels are already covered by other retained crops. Uses predictions **only**, never ground truth.
- `grid_128` / `grid_256`: fixed-grid proposal alternatives, select all tiles touched by a positive prediction. This tests a substantially different ROI algorithm.
- `random_0...`: equal-size, equal-count random-position crops as a **negative control**, never eligible as a recommended model/policy. If random crops recover almost as much GT, the generator may be forwarding too much area.

Merge, prune and grid are isolated **experimental prototypes**, not replacements for production `satchange` functions. They never access GT to construct boxes; GT is used only afterward for evaluation.

In [4]:
_DUMMY_IMAGES = {}
def dummy_rgb(shape):
    if shape not in _DUMMY_IMAGES:
        _DUMMY_IMAGES[shape]=torch.zeros((3,*shape),dtype=torch.float32)
    return _DUMMY_IMAGES[shape]

def project_boxes(binary, min_crop_size=128):
    h,w=binary.shape
    probabilities=torch.from_numpy(binary[np.newaxis].astype(np.float32,copy=True))
    # Binary 0/1 probabilities with threshold=0.5 give identical components to thresholded score maps.
    with torch.inference_mode():
        rois=extract_rois(dummy_rgb((h,w)),dummy_rgb((h,w)),probabilities,
            settings=ROISettings(threshold=0.5,connectivity=8,context_pixels=CONTEXT_PX,
              min_crop_size=int(min_crop_size),min_component_area=MIN_COMPONENT_AREA))
    return [tuple(map(int,r.crop_bbox)) for r in rois]

def box_area(b): return (b[2]-b[0])*(b[3]-b[1])
def intersection_area(a,b):
    return max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))

def merge_overlapping_boxes(boxes, shape, fraction=MERGE_OVERLAP_FRACTION):
    # Deterministic iterative union merging, with an explicit hard cap on merged area.
    out=sorted(set(map(tuple,boxes)))
    cap=MAX_MERGED_AREA_FRACTION*int(np.prod(shape))
    while True:
        best=None
        for i in range(len(out)):
            for j in range(i+1,len(out)):
                overlap=intersection_area(out[i],out[j])
                if overlap==0 or overlap/min(box_area(out[i]),box_area(out[j])) < fraction:
                    continue
                u=(min(out[i][0],out[j][0]),min(out[i][1],out[j][1]),
                   max(out[i][2],out[j][2]),max(out[i][3],out[j][3]))
                if box_area(u)>cap:continue
                best=(i,j,u);break
            if best is not None:break
        if best is None:return out
        i,j,u=best
        out=[b for k,b in enumerate(out) if k not in (i,j)]+[u]
        out=sorted(out)

def prune_redundant_boxes(boxes,binary):
    # Keep only proposals adding unseen predicted positive pixels. No GT is read.
    if not boxes:return []
    positives=np.asarray(binary,dtype=bool)
    covered=np.zeros_like(positives)
    remaining=list(range(len(boxes)))
    kept=[]
    while remaining:
        scores=[]
        for idx in remaining:
            x0,y0,x1,y1=boxes[idx]
            extra=int(np.count_nonzero(positives[y0:y1,x0:x1] & ~covered[y0:y1,x0:x1]))
            scores.append((extra/max(1,box_area(boxes[idx])),extra,-box_area(boxes[idx]),-idx,idx))
        pick=max(scores)[-1]
        if max(scores)[1]==0:break
        kept.append(boxes[pick]);remaining.remove(pick)
        x0,y0,x1,y1=boxes[pick]
        covered[y0:y1,x0:x1]=True
    # No positive predicted pixel covered by an original crop is lost.
    mask_original=np.zeros_like(positives)
    for x0,y0,x1,y1 in boxes:mask_original[y0:y1,x0:x1]=True
    if np.any(positives & mask_original & ~covered):
        raise AssertionError('Pruning lost predicted-positive coverage')
    return kept

def grid_proposal_boxes(binary,side):
    h,w=binary.shape;boxes=[]
    for y0 in range(0,h,side):
        y1=min(h,y0+side)
        for x0 in range(0,w,side):
            x1=min(w,x0+side)
            if binary[y0:y1,x0:x1].any():boxes.append((x0,y0,x1,y1))
    return boxes

def random_position_boxes(boxes,shape,seed):
    rng=np.random.default_rng(int(seed));h,w=shape;out=[]
    for x0,y0,x1,y1 in boxes:
        bh,bw=y1-y0,x1-x0
        nx=int(rng.integers(0,w-bw+1));ny=int(rng.integers(0,h-bh+1))
        out.append((nx,ny,nx+bw,ny+bh))
    return out

def verify_boxes(boxes,shape):
    h,w=shape
    for box in boxes:
        x0,y0,x1,y1=box
        if not (0<=x0<x1<=w and 0<=y0<y1<=h):
            raise AssertionError(f'Invalid proposal {box} in native shape {shape}')
    return boxes

def all_roi_proposals(binary,variant,threshold,index):
    base=project_boxes(binary,BASE_CROP_SIZE)
    seedbase=BOOTSTRAP_SEED + index*10_000 + (0 if 'medium' in variant else 317)
    out={
        'project_128':base,
        'project_64':project_boxes(binary,SMALL_CROP_SIZE),
        'merge_overlaps':merge_overlapping_boxes(base,binary.shape),
        'prune_redundant':prune_redundant_boxes(base,binary),
        'grid_128':grid_proposal_boxes(binary,128),
        'grid_256':grid_proposal_boxes(binary,256),
    }
    for replicate in range(RANDOM_CONTROL_REPEATS):
        out[f'random_{replicate}']=random_position_boxes(base,binary.shape,seedbase+replicate)
    return {name:verify_boxes(boxes,binary.shape) for name,boxes in out.items()}

def evaluate_rois(boxes,gt,pred_binary):
    shape=gt['mask'].shape;union=np.zeros(shape,dtype=bool)
    total=0;false=0;gt_dense=[]
    for x0,y0,x1,y1 in boxes:
        union[y0:y1,x0:x1]=True
        area=(x1-x0)*(y1-y0); total+=area
        count=int(gt['mask'][y0:y1,x0:x1].sum())
        false+=(count==0);gt_dense.append(count/area)
    gl=gt['labels'];n_gt=gt['n_gt']
    sizes=gt['areas'][1:]
    pix_cov=np.bincount(gl[union],minlength=n_gt+1)[1:]
    raw_cov=np.bincount(gl[pred_binary],minlength=n_gt+1)[1:]
    rate=np.divide(pix_cov,sizes,out=np.zeros_like(pix_cov,dtype=float),where=sizes>0)
    raw_rate=np.divide(raw_cov,sizes,out=np.zeros_like(raw_cov,dtype=float),where=sizes>0)
    return dict(
        n_gt=n_gt,n_gt_pixels=int(gt['mask'].sum()),gt_captured_pixels=int(np.count_nonzero(union&gt['mask'])),
        gt_covered_95=int(np.count_nonzero(rate>=.95)),gt_missed_zero=int(np.count_nonzero(rate==0)),
        rescued_from_zero=int(np.count_nonzero((raw_rate==0)&(rate>=.95))),
        rescued_from_below50=int(np.count_nonzero((raw_rate<.5)&(rate>=.95))),
        n_crops=len(boxes),false_crops=false,total_crop_pixels=total,
        union_pixels=int(union.sum()),image_pixels=int(np.prod(shape)),
        mean_gt_density_per_crop=float(np.mean(gt_dense)) if gt_dense else 0.0,
    )

# Synthetic guarantees of alternative algorithms. Run before any real data.
_s=np.zeros((256,256),dtype=bool);_s[62:68,62:68]=True;_s[90:95,92:96]=True
_b=project_boxes(_s)
assert len(_b)==2, 'Project ROI extraction unexpected on synthetic mask.'
# Prove binary-score proposal geometry matches the original project's direct probability-threshold call.
_test_prob=np.where(_s,0.8,0.1).astype(np.float32)
_direct=extract_rois(dummy_rgb(_s.shape),dummy_rgb(_s.shape),
    torch.from_numpy(_test_prob[None]),
    settings=ROISettings(threshold=0.35,connectivity=8,context_pixels=CONTEXT_PX,
                         min_crop_size=BASE_CROP_SIZE,min_component_area=MIN_COMPONENT_AREA))
assert [tuple(map(int,r.crop_bbox)) for r in _direct]==_b, 'Binarized input changed original ROI geometry'
assert len(merge_overlapping_boxes(_b,_s.shape))<=len(_b)
assert len(prune_redundant_boxes(_b,_s))<=len(_b)
assert sum(box_area(x) for x in random_position_boxes(_b,_s.shape,11))==sum(map(box_area,_b))
assert len(grid_proposal_boxes(_s,128))>=1
print('Synthetic ROI algorithm checks: passed')

Synthetic ROI algorithm checks: passed


## 4. ROI experiment — same masks, alternative crop generators

Each pair is evaluated with the same original GT component labels and the same cached pixel score map. Exports include per-pair results for **paired** comparisons; aggregated coverage is computed from sums over components or pixels, never an unweighted mean of per-image percentages.

In [5]:
roi_rows=[]
if RUN_ROI_ALTERNATIVES:
    for variant in ROI_MODELS:
        spec=variant_specs[variant]
        for threshold in ROI_THRESHOLDS:
            for i in range(ns):
                gt=experiment.gt('val',i)
                score=experiment.score(spec,'val',i)
                binary=score>=np.float32(threshold)
                tic=time.perf_counter()
                proposals=all_roi_proposals(binary,variant,threshold,i)
                proposal_ms=(time.perf_counter()-tic)*1000
                for method,boxes in proposals.items():
                    m=evaluate_rois(boxes,gt,binary)
                    roi_rows.append(dict(candidate=variant,threshold=float(threshold),method=method,
                        pair_index=i,sample_id=gt['sample_id'],internal_split=pair_split[i],
                        proposal_group_ms=proposal_ms,**m))
            print('ROI generated:',variant,'threshold',threshold,'pairs',ns,flush=True)
    roi_pair=pd.DataFrame(roi_rows)
    if roi_pair.duplicated(['candidate','threshold','method','pair_index']).any():
        raise RuntimeError('Duplicate per-pair ROI results')
    roi_pair.to_csv(OUTPUT/'10_roi_alternatives_per_pair.csv',index=False)
    print('ROI alternatives:',len(roi_pair),'pair-rows')
else:roi_pair=pd.DataFrame()

def aggregate_roi(df):
    groups=[]
    for keys,g in df.groupby(['candidate','threshold','method','internal_split'],sort=True):
        comps=g.n_gt.sum();pixels=g.n_gt_pixels.sum()
        groups.append(dict(zip(['candidate','threshold','method','internal_split'],keys),
          pairs=len(g),n_gt=int(comps),roi_coverage95=g.gt_covered_95.sum()/comps,
          roi_pixel_recall=g.gt_captured_pixels.sum()/pixels,
          roi_completely_missed=g.gt_missed_zero.sum()/comps,
          rescued_zero=int(g.rescued_from_zero.sum()),
          crops_per_pair=g.n_crops.mean(),false_crops_per_pair=g.false_crops.mean(),
          false_crop_fraction=g.false_crops.sum()/max(1,g.n_crops.sum()),
          processed_area=g.total_crop_pixels.sum()/g.image_pixels.sum(),
          union_area=g.union_pixels.sum()/g.image_pixels.sum(),
          mean_crop_density=g.mean_gt_density_per_crop.mean()))
    return pd.DataFrame(groups)

if not roi_pair.empty:
    roi_summary=aggregate_roi(roi_pair)
    roi_summary.to_csv(OUTPUT/'10_roi_alternatives_summary.csv',index=False)
    display(roi_summary[roi_summary.candidate=='medium_full'][
        ['threshold','method','internal_split','roi_coverage95','roi_pixel_recall','crops_per_pair',
         'processed_area','union_area','false_crop_fraction']].round(4))
else:roi_summary=pd.DataFrame()

ROI generated: medium_full threshold 0.35 pairs 64
ROI generated: medium_full threshold 0.7 pairs 64
ROI generated: light_full threshold 0.35 pairs 64
ROI generated: light_full threshold 0.7 pairs 64
ROI generated: medium_resized threshold 0.35 pairs 64
ROI generated: medium_resized threshold 0.7 pairs 64
ROI generated: light_resized threshold 0.35 pairs 64
ROI generated: light_resized threshold 0.7 pairs 64
ROI alternatives: 4608 pair-rows


,threshold,method,internal_split,roi_coverage95,roi_pixel_recall,crops_per_pair,processed_area,union_area,false_crop_fraction
72,0.35,grid_128,audit,0.9831,0.9922,17.8438,0.2788,0.2788,0.1121
73,0.35,grid_128,dev,0.9780,0.9870,19.3750,0.3027,0.3027,0.1000
74,0.35,grid_256,audit,0.9935,0.9976,7.2500,0.4531,0.4531,0.0905
75,0.35,grid_256,dev,0.9890,0.9890,7.4062,0.4629,0.4629,0.1139
76,0.35,merge_overlaps,audit,0.9839,0.9931,6.1562,0.3015,0.2695,0.1472
77,0.35,merge_overlaps,dev,0.9774,0.9869,5.7812,0.3425,0.2991,0.2108
78,0.35,project_128,audit,0.9798,0.9914,34.3438,0.5395,0.2308,0.0555
79,0.35,project_128,dev,0.9716,0.9863,34.4375,0.5569,0.2550,0.0535
80,0.35,project_64,audit,0.9363,0.9752,34.3438,0.1451,0.1121,0.0910
81,0.35,project_64,dev,0.9438,0.9786,34.4375,0.1687,0.1363,0.0644


## 5. Choose proposals on development only; compare on internal audit

For each model, select the least processed-area nonrandom ROI method satisfying the **predeclared target ROI Component Coverage** on 32 development pairs. If none is feasible, choose the maximum-coverage method and flag it. Compare its internal-audit performance with the original ROI generator using the **same threshold**. Also report Light vs Medium operating points chosen independently under the same candidate grid/targets; check whether coverage is actually matched (tolerance 0.5 percentage points).

In [6]:
def coverage_target(variant):return TARGET_COVERAGE_95['resized' if 'resized' in variant else 'full']
def choose_dev(sub,model,methods=None):
    pool=sub[(sub.internal_split=='dev')&(sub.candidate==model)&~sub.method.str.startswith('random_')].copy()
    if methods is not None:pool=pool[pool.method.isin(methods)]
    if pool.empty:return None
    eligible=pool[pool.roi_coverage95>=coverage_target(model)]
    feasible=not eligible.empty
    if not feasible:
        selected=pool.sort_values(['roi_coverage95','processed_area','crops_per_pair'],ascending=[False,True,True]).iloc[0]
    else:
        selected=eligible.sort_values(['processed_area','crops_per_pair','method','threshold']).iloc[0]
    return dict(candidate=model,threshold=float(selected.threshold),method=str(selected.method),
        dev_coverage95=float(selected.roi_coverage95),dev_processed_area=float(selected.processed_area),
        requested_target=coverage_target(model),target_feasible=bool(feasible))

roi_choices=[];roi_audit=pd.DataFrame()
if not roi_summary.empty:
    for model in ROI_MODELS:
        chosen=choose_dev(roi_summary,model)
        if chosen is not None:roi_choices.append(chosen)
    choices=pd.DataFrame(roi_choices)
    choices.to_csv(OUTPUT/'10_roi_dev_selected_policies.csv',index=False)
    audit=roi_summary[roi_summary.internal_split=='audit'].copy()
    roi_audit=choices.merge(audit,on=['candidate','method','threshold'],how='left',validate='one_to_one')
    roi_audit.to_csv(OUTPUT/'10_roi_selected_audit_summary.csv',index=False)
    display(roi_audit[['candidate','method','threshold','requested_target','target_feasible',
                       'roi_coverage95','crops_per_pair','processed_area','false_crop_fraction']].round(4))

    def paired_roi_delta(a,b,reps=BOOTSTRAP_REPS,seed=BOOTSTRAP_SEED):
        aa=a.sort_values('pair_index').reset_index(drop=True)
        bb=b.sort_values('pair_index').reset_index(drop=True)
        if not np.array_equal(aa.sample_id.values,bb.sample_id.values):
            raise ValueError('Sample IDs misaligned; paired comparison cannot proceed.')
        rng=np.random.default_rng(seed);n=len(aa)
        draw=rng.integers(0,n,size=(reps,n))
        metrics={
          'roi_coverage95':('gt_covered_95','n_gt'),
          'roi_pixel_recall':('gt_captured_pixels','n_gt_pixels'),
          'crops_per_pair':('n_crops',None),
          'processed_area':('total_crop_pixels','image_pixels'),
          'union_area':('union_pixels','image_pixels'),
          'false_crops_per_pair':('false_crops',None),
        }
        records=[]
        for metric,(num,den) in metrics.items():
            xa=aa[num].to_numpy(float);xb=bb[num].to_numpy(float)
            if den is None:
                point=(xb-xa).mean();boot=(xb-xa)[draw].mean(axis=1)
            else:
                d=aa[den].to_numpy(float)
                if not np.array_equal(aa[den].to_numpy(),bb[den].to_numpy()):
                    raise ValueError('Mismatch in shared denominator '+den)
                point=(xb.sum()-xa.sum())/d.sum()
                boot=(xb[draw].sum(axis=1)-xa[draw].sum(axis=1))/d[draw].sum(axis=1)
            lo,hi=np.quantile(boot,[0.025,0.975])
            records.append(dict(metric=metric,delta=float(point),ci95_low=float(lo),ci95_high=float(hi)))
        return pd.DataFrame(records)

    audit_pairs=roi_pair[roi_pair.internal_split=='audit']
    ci_records=[]
    for c in roi_choices:
        model,t,method=c['candidate'],c['threshold'],c['method']
        orig=audit_pairs[(audit_pairs.candidate==model)&(audit_pairs.threshold==t)&(audit_pairs.method=='project_128')]
        alt=audit_pairs[(audit_pairs.candidate==model)&(audit_pairs.threshold==t)&(audit_pairs.method==method)]
        ci=paired_roi_delta(orig,alt);ci['candidate']=model;ci['comparison']=f'{method} − project_128 at threshold {t}'
        ci_records.append(ci)
    # Also evaluate independently selected Light and Medium policies at the same per-family target.
    for suffix in ('full','resized'):
        al=next((c for c in roi_choices if c['candidate']==f'light_{suffix}'),None)
        bm=next((c for c in roi_choices if c['candidate']==f'medium_{suffix}'),None)
        if al and bm:
            la=audit_pairs[(audit_pairs.candidate==al['candidate'])&(audit_pairs.method==al['method'])&
                (audit_pairs.threshold==al['threshold'])]
            me=audit_pairs[(audit_pairs.candidate==bm['candidate'])&(audit_pairs.method==bm['method'])&
                (audit_pairs.threshold==bm['threshold'])]
            ci=paired_roi_delta(la,me)
            cv=ci[ci.metric=='roi_coverage95'].delta.iloc[0]
            ci['candidate']=f'medium_{suffix} minus light_{suffix}'
            ci['comparison']='two independently dev-selected operating points'
            ci['coverage_match_abs_le_0p005']=abs(cv)<=COVERAGE_MATCH_TOLERANCE
            ci_records.append(ci)
    roi_ci=pd.concat(ci_records,ignore_index=True)
    roi_ci.to_csv(OUTPUT/'10_roi_audit_paired_bootstrap.csv',index=False)
    display(roi_ci.round(5))
else:
    choices=pd.DataFrame();roi_ci=pd.DataFrame()

,candidate,method,threshold,requested_target,target_feasible,roi_coverage95,crops_per_pair,processed_area,false_crop_fraction
0,medium_full,grid_128,0.7,0.975,True,0.9710,16.9062,0.2642,0.0832
1,light_full,grid_128,0.7,0.975,True,0.9758,18.0625,0.2822,0.1280
2,medium_resized,merge_overlaps,0.7,0.935,True,0.8758,7.1875,0.2270,0.1609
3,light_resized,grid_128,0.7,0.935,True,0.9306,16.1562,0.2524,0.1122


,metric,delta,ci95_low,ci95_high,candidate,comparison,coverage_match_abs_le_0p005
0,roi_coverage95,0.00403,-0.00384,0.01169,medium_full,grid_128 − project_128 at threshold 0.7,NaN
1,roi_pixel_recall,0.00243,0.00001,0.00570,medium_full,grid_128 − project_128 at threshold 0.7,NaN
2,crops_per_pair,-18.96875,-29.93828,-9.68594,medium_full,grid_128 − project_128 at threshold 0.7,NaN
3,processed_area,-0.29761,-0.46920,-0.15179,medium_full,grid_128 − project_128 at threshold 0.7,NaN
4,union_area,0.04375,0.03049,0.05867,medium_full,grid_128 − project_128 at threshold 0.7,NaN
5,false_crops_per_pair,0.09375,-0.25078,0.43750,medium_full,grid_128 − project_128 at threshold 0.7,NaN
6,roi_coverage95,0.01371,0.00450,0.02753,light_full,grid_128 − project_128 at threshold 0.7,NaN
7,roi_pixel_recall,0.00449,0.00135,0.00928,light_full,grid_128 − project_128 at threshold 0.7,NaN
8,crops_per_pair,-23.65625,-36.28281,-12.78125,light_full,grid_128 − project_128 at threshold 0.7,NaN
9,processed_area,-0.37124,-0.56754,-0.20149,light_full,grid_128 − project_128 at threshold 0.7,NaN


## 6. Topology ablations — changes to the binary segmentation mask, not ROI boxes

These are **post-processing** interventions on the same model scores, *not* evidence about training hyperparameters. No GT drives any of these algorithms. They may remove spurious thin bridges, split merged blobs, or remove noise, but can also delete true positives. We always report Pixel F1 and one-to-one Component Recall together, plus merging counts and completely missed GT components.

- `raw`: unchanged thresholded mask.
- `open3` / `open5`: morphological opening can disconnect narrow bridges but may delete small buildings.
- `close3`: fills small gaps, may help fragmentation but can worsen merging.
- `open3_close3`: combined smooth/sharpening check.
- `remove_small16`: rejects islands smaller than 16 pixels.
- `edt_split`: experimental watershed splitting of connected regions at EDT ridges, optional scikit-image; preserves raw positive pixels except separator lines. It does **not** use GT, and must be evaluated for false splits.

**Topology measurement** uses the original 06 `_evaluate_mask` and matching implementation. It is *not* reimplemented here.

In [7]:
try:
    _WATER = watershed
    _PEAKS = peak_local_max
except NameError:
    _WATER = None
    _PEAKS = None

def distance_watershed_split(binary,min_distance=14):
    if _WATER is None or _PEAKS is None:
        raise RuntimeError('skimage watershed not available')
    if not binary.any():return binary.copy()
    distance=ndi.distance_transform_edt(binary)
    peaks=_PEAKS(distance,min_distance=min_distance,threshold_abs=2.0,
                 labels=binary.astype(np.uint8),exclude_border=False)
    markers=np.zeros(binary.shape,dtype=np.int32)
    for k,(y,x) in enumerate(peaks,1):markers[y,x]=k
    # Ensure every original connected component has at least one seed.
    components,ncomp=ndi.label(binary,structure=np.ones((3,3),dtype=bool))
    has_marker=np.bincount(components[markers>0],minlength=ncomp+1)>0
    missing_ids=np.flatnonzero(~has_marker[1:])+1
    for comp_id in missing_ids:
        ys,xs=np.nonzero(components==comp_id)
        if len(ys):
            ix=int(np.argmax(distance[ys,xs]));markers[ys[ix],xs[ix]]=int(markers.max())+1
    segments=_WATER(-distance,markers,mask=binary,watershed_line=True,
                     connectivity=np.ones((3,3),dtype=bool))
    result=(segments>0)&binary
    if not np.all(~result|binary):raise AssertionError('Watershed added positive pixels')
    return result

def topology_variants(binary):
    b=np.asarray(binary,dtype=np.uint8)
    k3=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    k5=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
    open3=cv2.morphologyEx(b,cv2.MORPH_OPEN,k3)
    variants={
      'raw': b.astype(bool),
      'open3':open3.astype(bool),
      'open5':cv2.morphologyEx(b,cv2.MORPH_OPEN,k5).astype(bool),
      'close3':cv2.morphologyEx(b,cv2.MORPH_CLOSE,k3).astype(bool),
      'open3_close3':cv2.morphologyEx(open3,cv2.MORPH_CLOSE,k3).astype(bool),
    }
    lab,n=ndi.label(binary,structure=np.ones((3,3),bool))
    areas=np.bincount(lab.ravel(),minlength=n+1);keep=(areas>=16);keep[0]=False
    variants['remove_small16']=keep[lab]
    if RUN_WATERSHED and WATERSHED_AVAILABLE:
        variants['edt_split']=distance_watershed_split(binary)
    return variants

def topology_metrics(binary,gt):
    # Importantly, use exact original H1–H5 matching and labels.
    counts,gt_info=experiment._evaluate_mask(binary,gt,
        list(experiment.plan['size_cutoffs_px']) if experiment.plan else [100,1000],geometry=False)
    pl,npred=ndi.label(binary,structure=experiment.gt_structure)
    gl=gt['labels'];valid=(pl>0)&(gl>0)
    if valid.any():
        pairs=np.unique(np.stack([gl[valid],pl[valid]],axis=1),axis=0)
        per_pred=np.bincount(pairs[:,1],minlength=npred+1)[1:]
        multi=int(np.count_nonzero(per_pred>1))
    else:multi=0
    return dict(tp=int(counts['tp']),fp=int(counts['fp']),fn=int(counts['fn']),
        n_gt=int(gt['n_gt']),n_pred=int(npred),obj_matches_050=int(counts['obj_tp_050']),
        gt_completely_missed=int(counts['completely_missed_gt']),n_pred_multi_gt=multi,
        pixel_pred_pos=int(binary.sum()))

# Synthetic masks: topology changes cannot be confused with ROI-only actions.
_v=np.zeros((80,80),bool);_v[10:25,10:25]=True;_v[40:60,40:60]=True;_v[20:45,22:24]=True
_t=topology_variants(_v)
assert np.array_equal(_t['raw'],_v)
assert all(x.shape==_v.shape and x.dtype==bool for x in _t.values())
assert not np.any(_t['edt_split'] & ~_v) if 'edt_split' in _t else True
print('Topology algorithm checks:',list(_t))

Topology algorithm checks: ['raw', 'open3', 'open5', 'close3', 'open3_close3', 'remove_small16']


## 7. Measure changes to Medium/Light segmentation topology on validation

In [8]:
topo_rows=[]
if RUN_TOPOLOGY_ABLATIONS:
    for variant in TOPOLOGY_MODELS:
        spec=variant_specs[variant]
        for threshold in TOPOLOGY_THRESHOLDS:
            for i in range(ns):
                gt=experiment.gt('val',i)
                original=experiment.score(spec,'val',i)>=np.float32(threshold)
                tic=time.perf_counter()
                variants=topology_variants(original)
                transform_total_ms=(time.perf_counter()-tic)*1000
                for name,binary in variants.items():
                    metrics=topology_metrics(binary,gt)
                    topo_rows.append(dict(candidate=variant,threshold=float(threshold),method=name,
                        pair_index=i,sample_id=gt['sample_id'],internal_split=pair_split[i],
                        postprocess_group_ms=transform_total_ms,**metrics))
            print('Topology done:',variant,'threshold',threshold,'pairs',ns,flush=True)
    topo_pair=pd.DataFrame(topo_rows)
    topo_pair.to_csv(OUTPUT/'10_topology_per_pair.csv',index=False)
else:topo_pair=pd.DataFrame()

def aggregate_topology(data):
    out=[]
    for keys,g in data.groupby(['candidate','threshold','method','internal_split'],sort=True):
        tp,fp,fn=g.tp.sum(),g.fp.sum(),g.fn.sum()
        out.append(dict(zip(['candidate','threshold','method','internal_split'],keys),
            pairs=len(g),n_gt=int(g.n_gt.sum()),pixel_f1=2*tp/max(1,2*tp+fp+fn),
            pixel_precision=tp/max(1,tp+fp),pixel_recall=tp/max(1,tp+fn),
            component_recall=g.obj_matches_050.sum()/g.n_gt.sum(),
            completely_missed_fraction=g.gt_completely_missed.sum()/g.n_gt.sum(),
            pred_multi_gt_fraction=g.n_pred_multi_gt.sum()/max(1,g.n_pred.sum()),
            mean_predicted_components=g.n_pred.mean(),mean_pred_positive_pixels=g.pixel_pred_pos.mean()))
    return pd.DataFrame(out)
if not topo_pair.empty:
    topo_summary=aggregate_topology(topo_pair)
    topo_summary.to_csv(OUTPUT/'10_topology_summary.csv',index=False)
    display(topo_summary[(topo_summary.candidate=='medium_full')&(topo_summary.internal_split=='audit')][
        ['threshold','method','pixel_f1','component_recall','pred_multi_gt_fraction',
         'completely_missed_fraction']].round(4))
else:topo_summary=pd.DataFrame()

Topology done: medium_full threshold 0.35 pairs 64
Topology done: medium_full threshold 0.7 pairs 64
Topology done: medium_full threshold 0.95 pairs 64
Topology done: light_full threshold 0.35 pairs 64
Topology done: light_full threshold 0.7 pairs 64
Topology done: light_full threshold 0.95 pairs 64
Topology done: medium_resized threshold 0.35 pairs 64
Topology done: medium_resized threshold 0.7 pairs 64
Topology done: medium_resized threshold 0.95 pairs 64
Topology done: light_resized threshold 0.35 pairs 64
Topology done: light_resized threshold 0.7 pairs 64
Topology done: light_resized threshold 0.95 pairs 64


,threshold,method,pixel_f1,component_recall,pred_multi_gt_fraction,completely_missed_fraction
72,0.35,close3,0.8306,0.5621,0.1238,0.0927
74,0.35,open3,0.8309,0.5903,0.1158,0.0952
76,0.35,open3_close3,0.8307,0.5629,0.1243,0.0952
78,0.35,open5,0.8311,0.5984,0.1154,0.1032
80,0.35,raw,0.8309,0.5879,0.1165,0.0927
82,0.35,remove_small16,0.8309,0.5879,0.1200,0.0992
84,0.70,close3,0.8351,0.6669,0.0762,0.1121
86,0.70,open3,0.8351,0.6919,0.0675,0.1145
88,0.70,open3_close3,0.8351,0.6758,0.0729,0.1145
90,0.70,open5,0.8350,0.7000,0.0677,0.1242


## 8. Paired topology changes on internal audit (preselected on development)

Within each model + threshold, compare a topology intervention with the **raw mask at the same threshold**. Select on dev based on Component Recall with **Pixel F1 loss no larger than 0.01 absolute**. If no improvement, retain raw. Audit bootstrap for the paired difference is exploratory and must not be interpreted as an independent test.

In [9]:
topo_choices=[];topo_ci=pd.DataFrame()
if not topo_summary.empty:
    dev=topo_summary[topo_summary.internal_split=='dev']
    for (candidate,thr),g in dev.groupby(['candidate','threshold']):
        raw=g[g.method=='raw'].iloc[0]
        acceptable=g[g.pixel_f1>=raw.pixel_f1-.01]
        eligible=acceptable[acceptable.component_recall>raw.component_recall]
        if eligible.empty:best=raw
        else:best=eligible.sort_values(['component_recall','pixel_f1','method'],
                                          ascending=[False,False,True]).iloc[0]
        topo_choices.append(dict(candidate=candidate,threshold=thr,method=best.method,
            dev_component_recall=best.component_recall,dev_pixel_f1=best.pixel_f1,
            raw_dev_component_recall=raw.component_recall,raw_dev_pixel_f1=raw.pixel_f1))
    topo_selected=pd.DataFrame(topo_choices)
    topo_selected.to_csv(OUTPUT/'10_topology_dev_selection.csv',index=False)

    def topology_pair_bootstrap(raw,alt,reps=BOOTSTRAP_REPS,seed=BOOTSTRAP_SEED):
        raw=raw.sort_values('pair_index').reset_index(drop=True)
        alt=alt.sort_values('pair_index').reset_index(drop=True)
        if not np.array_equal(raw.sample_id.values,alt.sample_id.values):raise ValueError('Nonaligned topology pairs')
        rng=np.random.default_rng(seed);draw=rng.integers(0,len(raw),(reps,len(raw)))
        g=raw.n_gt.to_numpy(float)
        delta_match=alt.obj_matches_050.to_numpy(float)-raw.obj_matches_050.to_numpy(float)
        component=delta_match.sum()/g.sum()
        bc=delta_match[draw].sum(axis=1)/g[draw].sum(axis=1)
        def f1(sub):
            tp=sub.tp.to_numpy(float);fp=sub.fp.to_numpy(float);fn=sub.fn.to_numpy(float)
            a,b,c=tp[draw].sum(axis=1),fp[draw].sum(axis=1),fn[draw].sum(axis=1)
            return 2*tp.sum()/max(1,2*tp.sum()+fp.sum()+fn.sum()),2*a/np.maximum(1,2*a+b+c)
        f0,p0=f1(raw);f1a,p1=f1(alt)
        out=[]
        for metric,point,boot in [('component_recall',component,bc),('pixel_f1',f1a-f0,p1-p0)]:
            lo,hi=np.quantile(boot,[.025,.975]);out.append(dict(metric=metric,delta=point,ci95_low=lo,ci95_high=hi))
        return pd.DataFrame(out)
    audit=topo_pair[topo_pair.internal_split=='audit']
    stats=[]
    for q in topo_choices:
        m,t,method=q['candidate'],q['threshold'],q['method']
        raw=audit[(audit.candidate==m)&(audit.threshold==t)&(audit.method=='raw')]
        alt=audit[(audit.candidate==m)&(audit.threshold==t)&(audit.method==method)]
        z=topology_pair_bootstrap(raw,alt)
        z['candidate']=m;z['threshold']=t;z['selected_method']=method
        stats.append(z)
    topo_ci=pd.concat(stats,ignore_index=True)
    topo_ci.to_csv(OUTPUT/'10_topology_audit_paired_bootstrap.csv',index=False)
    display(topo_selected.round(4))
    display(topo_ci.round(5))
else:topo_selected=pd.DataFrame()

,candidate,threshold,method,dev_component_recall,dev_pixel_f1,raw_dev_component_recall,raw_dev_pixel_f1
0,light_full,0.35,open5,0.6318,0.8507,0.6163,0.8495
1,light_full,0.70,open5,0.7171,0.8585,0.7016,0.8582
2,light_full,0.95,open5,0.7855,0.8421,0.7778,0.8443
3,light_resized,0.35,open5,0.2022,0.7577,0.1964,0.7574
4,light_resized,0.70,open5,0.3094,0.7650,0.2965,0.7652
5,light_resized,0.95,open5,0.4438,0.6802,0.4360,0.6831
6,medium_full,0.35,open5,0.4767,0.8701,0.4651,0.8698
7,medium_full,0.70,open5,0.6253,0.8773,0.6156,0.8773
8,medium_full,0.95,open5,0.7558,0.8572,0.7435,0.8578
9,medium_resized,0.35,raw,0.0717,0.7257,0.0717,0.7257


,metric,delta,ci95_low,ci95_high,candidate,threshold,selected_method
0,component_recall,0.01452,0.00418,0.02523,light_full,0.35,open5
1,pixel_f1,0.00114,0.00063,0.00167,light_full,0.35,open5
2,component_recall,0.00565,-0.00183,0.01435,light_full,0.70,open5
3,pixel_f1,0.00027,-0.00012,0.00068,light_full,0.70,open5
4,component_recall,-0.00645,-0.01099,-0.00289,light_full,0.95,open5
5,pixel_f1,-0.00163,-0.00223,-0.00111,light_full,0.95,open5
6,component_recall,0.01210,0.00290,0.02381,light_resized,0.35,open5
7,pixel_f1,0.00060,0.00041,0.00079,light_resized,0.35,open5
8,component_recall,0.01694,0.00691,0.02658,light_resized,0.70,open5
9,pixel_f1,0.00010,-0.00009,0.00029,light_resized,0.70,open5


## 9. Optional interaction: does topology change help ROI proposal quality?

ROI-only methods do not affect pixel/component F1. The following **separate** experiment applies the development-selected topology choice for each model at threshold `0.35`, then calls the existing `extract_rois` on the *modified binary mask* using the same crop settings. Compare raw vs modified crops. This tells us whether better strict component matching, if any, transfers to better ROIs.

In [10]:
interaction_rows=[]
if RUN_INTERACTION and not topo_selected.empty:
    for model in ROI_MODELS:
        select=topo_selected[(topo_selected.candidate==model)&(topo_selected.threshold==0.35)]
        if select.empty:continue
        method=str(select.iloc[0].method)
        spec=variant_specs[model]
        for i in range(ns):
            gt=experiment.gt('val',i)
            raw=experiment.score(spec,'val',i)>=np.float32(.35)
            transformed=topology_variants(raw)[method]
            for label,b in [('raw',raw),('selected_'+method,transformed)]:
                boxes=project_boxes(b,BASE_CROP_SIZE)
                metrics=evaluate_rois(boxes,gt,raw)  # raw mask recall held fixed as diagnostic denominator
                interaction_rows.append(dict(candidate=model,threshold=.35,topology_policy=label,
                    pair_index=i,sample_id=gt['sample_id'],internal_split=pair_split[i],**metrics))
        print('Topology × ROI interaction:',model,method,flush=True)
    interaction=pd.DataFrame(interaction_rows)
    interaction.to_csv(OUTPUT/'10_topology_roi_interaction_per_pair.csv',index=False)
    res=(interaction.groupby(['candidate','topology_policy','internal_split'])
         .agg(n_gt=('n_gt','sum'),gt_covered_95=('gt_covered_95','sum'),
              n_crops=('n_crops','mean'),processed_px=('total_crop_pixels','sum'),
              image_px=('image_pixels','sum')).reset_index())
    res['roi_coverage95']=res.gt_covered_95/res.n_gt
    res['processed_area']=res.processed_px/res.image_px
    res.to_csv(OUTPUT/'10_topology_roi_interaction_summary.csv',index=False)
    display(res[res.internal_split=='audit'][['candidate','topology_policy','roi_coverage95','n_crops','processed_area']].round(4))
else:interaction=pd.DataFrame()

Topology × ROI interaction: medium_full open5
Topology × ROI interaction: light_full open5
Topology × ROI interaction: medium_resized raw
Topology × ROI interaction: light_resized open5


,candidate,topology_policy,roi_coverage95,n_crops,processed_area
0,light_full,raw,0.9702,42.1875,0.6618
2,light_full,selected_open5,0.9637,37.7188,0.5919
4,light_resized,raw,0.9250,23.4688,0.3898
6,light_resized,selected_open5,0.9242,23.2188,0.3854
8,medium_full,raw,0.9798,34.3438,0.5395
10,medium_full,selected_open5,0.9710,33.3125,0.5234
12,medium_resized,raw,0.8823,13.8438,0.2932
14,medium_resized,selected_raw,0.8823,13.8438,0.2932


## 10. Visual checks and Pareto plots

Visualizations use predeclared validation example indices rather than selecting impressive-looking successes or failures after reviewing results. Masks are shown against GT on the original native pixel grid; all bounding boxes are generated from the predictions only.

In [11]:
if RUN_CASE_STUDIES:
    for variant in ('medium_full','light_full'):
        if variant not in ROI_MODELS:continue
        spec=variant_specs[variant]
        for i in CASE_PAIR_INDICES:
            if i>=ns:continue
            gt=experiment.gt('val',i)
            binary=experiment.score(spec,'val',i)>=np.float32(.35)
            proposals=all_roi_proposals(binary,variant,.35,i)
            compare_methods=['project_128','merge_overlaps','grid_256']
            fig,axs=plt.subplots(1,4,figsize=(16,4))
            axs[0].imshow(gt['mask'],cmap='Greens',vmin=0,vmax=1)
            axs[0].imshow(binary,cmap='Reds',vmin=0,vmax=1,alpha=.4)
            axs[0].set_title('GT green + raw prediction red')
            for ax,method in zip(axs[1:],compare_methods):
                ax.imshow(gt['mask'],cmap='Greys',vmin=0,vmax=1,alpha=.5)
                for x0,y0,x1,y1 in proposals[method]:
                    ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,lw=.8,edgecolor='tab:blue'))
                ax.set_title(f'{method} ({len(proposals[method])} boxes)')
            for ax in axs:ax.set_axis_off()
            fig.suptitle(f'{variant} | validation pair {i} | source GT+prediction | no cherry-picking')
            fig.tight_layout();fig.savefig(OUTPUT/'figures'/f'10_roi_{variant}_pair_{i:03d}.png',dpi=140)
            plt.close(fig)
    if not roi_summary.empty:
        fig,ax=plt.subplots(figsize=(10,6))
        for (model,method),g in roi_summary[roi_summary.internal_split=='audit'].groupby(['candidate','method']):
            if method.startswith('random_'):continue
            ax.scatter(g.processed_area,g.roi_coverage95,label=f'{model}:{method}',s=22)
        ax.set(xlabel='Total processed crop area / source area (can exceed 1)',
               ylabel='Fraction of GT components >= 95% inside crop union',
               title='ROI quality–workload alternatives | internal audit')
        ax.grid(alpha=.2);ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=7)
        fig.tight_layout();fig.savefig(OUTPUT/'figures'/'10_roi_pareto_audit.png',dpi=160);plt.close(fig)
    if not topo_summary.empty:
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        sub=topo_summary[(topo_summary.internal_split=='audit')&(topo_summary.candidate=='medium_full')]
        for method,g in sub.groupby('method'):
            axs[0].plot(g.threshold,g.component_recall,marker='o',label=method)
            axs[1].plot(g.threshold,g.pixel_f1,marker='o',label=method)
        axs[0].set(title='Medium Full: strict Component Recall',xlabel='Threshold',ylabel='Recall')
        axs[1].set(title='Medium Full: Pixel F1',xlabel='Threshold',ylabel='F1')
        for ax in axs:ax.grid(alpha=.2)
        axs[1].legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8)
        fig.tight_layout();fig.savefig(OUTPUT/'figures'/'10_medium_topology_audit.png',dpi=160);plt.close(fig)
print('Figures saved under',OUTPUT/'figures')

Figures saved under artifacts/hypothesis_tests/H10_roi_generator_topology_ablations/figures


## 11. Decision table / interpretation guards

Do not conclude that opening or watershed has *retrained* the model: these are post-processing operations, and changes to Component Recall may be accompanied by lost true pixels. Do not infer that a random-position ROI baseline is a deployable proposal strategy. A model-specific optimized ROI strategy is a useful **upper-bound engineering investigation**, but a fair architecture comparison also needs identical search spaces, same development/audit split, and total system cost including semantic inference. Neither SECOND generalization nor WHAT accuracy is measured in this notebook.

In [12]:
decision_questions=[
('ROI algorithm effect','At the same model+threshold, does project_64 / merge / grid / prune change audit coverage and crop/area cost?',
 'Use paired audit contrasts, no GT-driven candidate proposals.'),
('Crop geometry confound','Does random-position coverage stay high at exactly matched crop count and sizes?',
 'Random control is diagnostic, never the selected model.'),
('Topology problem','Can opening / size removal / watershed raise strict one-to-one IoU>=0.5 recall?',
 'Inspect Pixel F1 and completely-missed GT simultaneously.'),
('Topology vs ROI','Do any topology changes translate to ROI gains under the original crop extractor?',
 'Report interaction separately; topology improvement alone does not ensure semantic improvement.'),
('Retraining decision','Do these results establish that Medium needs retraining?',
 'No: post-processing on LEVIR cannot identify training hyperparameter causality or SECOND transfer.'),
]
decision_frame=pd.DataFrame(decision_questions,columns=['question','evaluated_by','interpretation_guard'])
decision_frame.to_csv(OUTPUT/'10_decision_scope.csv',index=False)
display(decision_frame)

,question,evaluated_by,interpretation_guard
0,ROI algorithm effect,"At the same model+threshold, does project_64 /...","Use paired audit contrasts, no GT-driven candi..."
1,Crop geometry confound,Does random-position coverage stay high at exa...,"Random control is diagnostic, never the select..."
2,Topology problem,Can opening / size removal / watershed raise s...,Inspect Pixel F1 and completely-missed GT simu...
3,Topology vs ROI,Do any topology changes translate to ROI gains...,Report interaction separately; topology improv...
4,Retraining decision,Do these results establish that Medium needs r...,No: post-processing on LEVIR cannot identify t...


## 12. Export all results, configs and provenance to one ZIP

In [13]:
manifest={
 'run':OUT_NAME,'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_v1_signature':experiment.signature,'source_v1_core_sha256':core_sha,
 'project_roi_module_sha256':roi_sha,'roi_module_path':str(roi_module.__file__),
 'validation_only':True,'test_used':False,'training':False,'gpu_inference':False,
 'run_mode':RUN_MODE,'validation_pairs':ns,'dev_pair_indices':sorted(dev_indices),
 'audit_pair_indices':sorted(audit_indices),'audit_split_seed':INTERNAL_AUDIT_SEED,
 'roi_models':list(ROI_MODELS),'topology_models':list(TOPOLOGY_MODELS),
 'roi_thresholds':list(ROI_THRESHOLDS),'topology_thresholds':list(TOPOLOGY_THRESHOLDS),
 'topology_algorithms':sorted(topo_pair.method.unique().tolist()) if not topo_pair.empty else [],
 'roi_algorithms':sorted(roi_pair.method.unique().tolist()) if not roi_pair.empty else [],
 'target_coverage':TARGET_COVERAGE_95,'coverage_match_tolerance':COVERAGE_MATCH_TOLERANCE,
 'roi_context_pixels':CONTEXT_PX,'roi_min_component_area':MIN_COMPONENT_AREA,
 'min_crop_sizes':[BASE_CROP_SIZE,SMALL_CROP_SIZE],
 'merge_overlap_fraction':MERGE_OVERLAP_FRACTION,
 'max_merged_area_fraction':MAX_MERGED_AREA_FRACTION,
 'random_control_repeats':RANDOM_CONTROL_REPEATS,
 'bootstrap_repetitions':BOOTSTRAP_REPS,'bootstrap_seed':BOOTSTRAP_SEED,
 'skimage_watershed_available':WATERSHED_AVAILABLE,
 'run_flags':{'roi':RUN_ROI_ALTERNATIVES,'topology':RUN_TOPOLOGY_ABLATIONS,
              'interaction':RUN_INTERACTION,'case_studies':RUN_CASE_STUDIES,'watershed':RUN_WATERSHED},
 'limitations':[
   'Exploratory analysis of LEVIR-CD validation only; no SECOND domain evidence.',
   'Internal audit is not a new independent test dataset.',
   'No downstream WHAT model inference or measured end-to-end cost.',
   'New ROI algorithms are experiment-only postprocessors, not changes to the project implementation.',
   'Watershed requires scikit-image and is skipped if not installed.',
 ]
}
(OUTPUT/'10_run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
archive=OUTPUT/(OUT_NAME+'_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and path!=archive and path.suffix.lower() in ('.csv','.png','.json'):
            z.write(path,path.relative_to(OUTPUT).as_posix())
print('Results archive:',archive,'| MB:',round(archive.stat().st_size/(1024**2),2))
display(FileLink(str(archive)))

Results archive: artifacts/hypothesis_tests/H10_roi_generator_topology_ablations/H10_roi_generator_topology_ablations_results.zip | MB: 2.48


artifacts/hypothesis_tests/H10_roi_generator_topology_ablations/H10_roi_generator_topology_ablations_results.zip